# BI Flow — BI Auditor / XAI Agent (Google Colab)

**Multi-Agent Automation of the Power BI Pipeline — import / export trade data**
ESPRIT · 5DS · 2026/2027

This notebook implements the **BI Auditor / XAI agent** and everything it needs to run end to end.

| Part | Content |
|---|---|
| **A. Simulation** | The 7 upstream agents (Profiler, Cleaning, ETL, KPI, Analyst, Dashboard, Orchestrator) are **simulated** and write their outputs as **CSV files** into `bi_flow_outputs/`. The data is generated, then really cleaned / modelled / aggregated, so every log is consistent with the numbers. |
| **B. Auditor** | Reads **only the CSVs** (you can replace them with your real agents' outputs). Builds a unified audit trail, lineage (visual → KPI → columns → ETL → cleaning rule → profiler issue → raw file), an independent KPI reconciliation, a **real RAG** (ChromaDB + sentence-transformers) and a **real LLM** (local Hugging Face model, or Claude through the API) that writes the explanations. |

**Design rule:** facts come from deterministic records (logs, lineage, recomputation); the LLM only *phrases* them.
Every LLM answer is **verified** (cited evidence ids must exist, numbers must appear in the evidence) and retried if it fails.

**Outputs:** `audit_trail.csv`, `lineage.csv`, `reconciliation.csv`, `xai_explanations.csv`, `human_review_queue.csv`, `xai_report.md`.

> **Runtime:** use *Runtime → Change runtime type → T4 GPU* for the local LLM (CPU works but is slow).

In [ ]:
%pip install -q chromadb sentence-transformers accelerate transformers anthropic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [ ]:
import os, re, json, random, difflib, unicodedata, shutil, warnings, datetime as dt
from collections import defaultdict
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 30)

# ======================= CONFIGURATION =======================
SEED = 42
OUT = "bi_flow_outputs"            # every CSV / report is written here
N_BASE_ROWS = 600                  # size of the simulated raw trade file (before duplicates)

LLM_PROVIDER = "hf"                # "hf" = local Hugging Face model (free, no key) | "anthropic" = Claude API
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # try "Qwen/Qwen2.5-3B-Instruct" on a T4 for better prose
ANTHROPIC_MODEL = "claude-sonnet-5-5"     # needs ANTHROPIC_API_KEY in Colab secrets (key icon, left panel)
MAX_NEW_TOKENS = 280

# how many items the LLM explains (lower these on CPU)
N_CORR_EXPLAIN = 6                 # corrections (one per cleaning rule, human-validation ones first)
N_KPI_EXPLAIN = 5                  # KPIs
N_INSIGHT_EXPLAIN = 4              # insights (+ the justification of each recommendation)

os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(SEED)
random.seed(SEED)
T0 = dt.datetime(2026, 10, 28, 9, 0, 0)

def ts(seconds):
    return (T0 + dt.timedelta(seconds=float(seconds))).isoformat(timespec="seconds")

def save(df, name):
    df.to_csv(f"{OUT}/{name}", index=False)
    print(f"  saved {OUT}/{name:<28s} {len(df):>6} rows x {df.shape[1]:>2} cols")

def rd(name, num=()):
    """Read a CSV as text (keeps HS codes like 150910 intact), then cast the numeric columns."""
    df = pd.read_csv(f"{OUT}/{name}", dtype=str)
    for c in num:
        df[c] = pd.to_numeric(df[c])
    return df

print("Config OK.")

Config OK.


---
# PART A — Simulated outputs of the 7 upstream agents (written as CSV)

## A.1 Reference data and the raw (dirty) trade file

In [ ]:
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_txt(s):
    return re.sub(r"[.\s]+", "", strip_accents(str(s)).lower())

COUNTRIES = {
    "TN": ("Tunisia", "Africa"), "FR": ("France", "Europe"), "IT": ("Italy", "Europe"),
    "DE": ("Germany", "Europe"), "ES": ("Spain", "Europe"), "LY": ("Libya", "Africa"),
    "DZ": ("Algeria", "Africa"), "MA": ("Morocco", "Africa"), "US": ("United States", "Americas"),
    "CN": ("China", "Asia"), "TR": ("Turkey", "Asia"), "AE": ("United Arab Emirates", "Asia"),
    "GB": ("United Kingdom", "Europe"), "EG": ("Egypt", "Africa"), "SA": ("Saudi Arabia", "Asia"),
}
VARIANTS = {
    "TN": ["Tunisia", "Tunisie", "TUNISIA", "TUN"], "FR": ["France", "FRANCE", "Fr."],
    "IT": ["Italy", "Italie", "ITALY"], "DE": ["Germany", "Allemagne", "Deutschland"],
    "ES": ["Spain", "Espagne", "SPAIN"], "LY": ["Libya", "Libye"], "DZ": ["Algeria", "Algérie", "Algerie"],
    "MA": ["Morocco", "Maroc"], "US": ["United States", "USA", "U.S.A", "États-Unis"],
    "CN": ["China", "Chine"], "TR": ["Turkey", "Turquie"],
    "AE": ["UAE", "United Arab Emirates", "Emirats Arabes Unis"],
    "GB": ["United Kingdom", "UK", "Royaume-Uni"], "EG": ["Egypt", "Egypte"], "SA": ["Saudi Arabia", "Arabie Saoudite"],
}
TYPOS = {"Frnace": "FR", "Germny": "DE", "Itally": "IT", "Algerai": "DZ",
         "Moroco": "MA", "Turkye": "TR", "Tunsia": "TN", "Egpyt": "EG"}
TYPO_BY_ISO = {v: k for k, v in TYPOS.items()}
ALIAS = {norm_txt(v): iso for iso, vs in VARIANTS.items() for v in vs + [iso, COUNTRIES[iso][0]]}

PRODUCTS = {
    "150910": "Virgin olive oil", "080410": "Dates, fresh or dried", "610910": "T-shirts, cotton, knitted",
    "854442": "Insulated electric conductors", "270900": "Crude petroleum oils", "100199": "Wheat and meslin",
    "300490": "Medicaments, retail packaged", "870323": "Passenger cars 1500-3000cc", "252329": "Portland cement",
}
EXPORT_PRODUCTS = ["150910", "080410", "610910", "854442"]
IMPORT_PRODUCTS = ["270900", "100199", "300490", "870323", "252329"]
BASE_EUR = {"150910": 60000, "080410": 25000, "610910": 18000, "854442": 40000, "270900": 90000,
            "100199": 45000, "300490": 35000, "870323": 28000, "252329": 15000}
EXP_MKT = {"FR": .28, "IT": .22, "DE": .15, "ES": .08, "LY": .10, "US": .06, "GB": .04, "AE": .04, "DZ": .03}
IMP_MKT = {"CN": .25, "IT": .15, "FR": .14, "TR": .12, "DE": .10, "DZ": .10, "ES": .05, "AE": .04, "EG": .03, "SA": .02}
CCY_BY_ISO = {"FR": "EUR", "IT": "EUR", "DE": "EUR", "ES": "EUR", "US": "USD", "CN": "USD", "AE": "USD", "SA": "USD"}
FX = {"EUR": 3.35, "USD": 3.10, "TND": 1.0}
CCY_VARIANTS = {"EUR": ["EUR", "eur", "€", "Euro"], "USD": ["USD", "usd", "$", "US$"], "TND": ["TND", "tnd", "DT", "Dinar"]}
FLOW_VARIANTS = {"Export": ["Export", "export", "EXPORT", "exp", "Exp."], "Import": ["Import", "import", "IMPORT", "imp", "Imp."]}
FLOW_MAP = {norm_txt(v): k for k, vs in FLOW_VARIANTS.items() for v in vs + [k]}
CCY_MAP = {norm_txt(v): k for k, vs in CCY_VARIANTS.items() for v in vs + [k]}
SOURCES = ["customs_declarations.csv", "erp_export.xlsx", "manual_sheet.xlsx"]

# ---- shared parsing helpers (used by the Profiler and the Cleaning agent) ----
def parse_date(s):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y"):
        try:
            return dt.datetime.strptime(str(s), fmt).date(), fmt
        except ValueError:
            pass
    return None, None

def hs_digits(s):
    return re.sub(r"\D", "", str(s))

def resolve_country(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return None, "missing", 0.0
    k = norm_txt(raw)
    if k in ALIAS:
        return ALIAS[k], "alias_lookup", 1.0
    m = difflib.get_close_matches(k, list(ALIAS), n=1, cutoff=0.75)
    if m:
        return ALIAS[m[0]], "fuzzy_match", round(difflib.SequenceMatcher(None, k, m[0]).ratio(), 2)
    return None, "unresolved", 0.0

# ---- generate the raw file ----
rows = []
for i in range(N_BASE_ROWS):
    flow = str(rng.choice(["Export", "Import"], p=[.55, .45]))
    pool = EXP_MKT if flow == "Export" else IMP_MKT
    iso = str(rng.choice(list(pool), p=np.array(list(pool.values())) / sum(pool.values())))
    hs = str(rng.choice(EXPORT_PRODUCTS if flow == "Export" else IMPORT_PRODUCTS))
    d = dt.date(int(rng.choice([2023, 2024])), int(rng.integers(1, 13)), int(rng.integers(1, 29)))
    eur = BASE_EUR[hs] * float(rng.lognormal(0, 0.45))
    if flow == "Export" and iso == "IT" and d.year == 2024: eur *= 0.60     # planted story: Italy drops
    if flow == "Export" and iso == "LY" and d.year == 2024: eur *= 1.40     # planted story: Libya grows
    if flow == "Export" and d.year == 2024 and d.month == 7: eur *= 2.50    # planted anomaly: July 2024
    ccy = CCY_BY_ISO.get(iso) or str(rng.choice(["EUR", "USD", "TND"]))
    value = round(eur * FX["EUR"] / FX[ccy], 2)
    qty = int(rng.integers(10, 5000))

    date_raw = [d.isoformat(), d.strftime("%d/%m/%Y"), d.strftime("%b %d, %Y")][int(rng.choice(3, p=[.5, .3, .2]))]
    r = rng.random()
    if r < .03: country_raw = np.nan
    elif r < .08 and iso in TYPO_BY_ISO: country_raw = TYPO_BY_ISO[iso]
    elif r < .25: country_raw = iso
    else: country_raw = str(rng.choice(VARIANTS[iso]))
    if rng.random() < .015:
        hs_raw = hs[:5]
    else:
        hs_raw = [hs, f"{hs[:4]}.{hs[4:]}", f"{hs[:4]} {hs[4:]}", f"{hs[:4]}-{hs[4:]}"][int(rng.choice(4, p=[.35, .35, .15, .15]))]
    if rng.random() < .01: value = -value
    rows.append(dict(
        declaration_id=f"DEC-{i:05d}", date_raw=date_raw, flow_raw=str(rng.choice(FLOW_VARIANTS[flow])),
        country_raw=country_raw, hs_code_raw=hs_raw, value_raw=value, currency_raw=str(rng.choice(CCY_VARIANTS[ccy])),
        quantity=qty, weight_kg=round(qty * float(rng.uniform(.5, 20)), 1),
        source_file=str(rng.choice(SOURCES, p=[.5, .35, .15]))))

raw_df = pd.DataFrame(rows)
dups = raw_df.sample(frac=.05, random_state=SEED)                      # duplicated declarations
raw_df = pd.concat([raw_df, dups]).sample(frac=1, random_state=SEED).reset_index(drop=True)
raw_df.insert(0, "row_id", range(1, len(raw_df) + 1))
print("Simulated agent outputs:")
save(raw_df, "raw_trade_data.csv")
save(pd.DataFrame([dict(currency=c, rate_to_tnd=r, rate_date="2024-12-31", source="simulated reference table")
                   for c, r in FX.items()]), "fx_rates.csv")
raw_df.head(8)

Simulated agent outputs:
  saved bi_flow_outputs/raw_trade_data.csv              630 rows x 11 cols
  saved bi_flow_outputs/fx_rates.csv                      3 rows x  4 cols


,row_id,declaration_id,date_raw,flow_raw,country_raw,hs_code_raw,value_raw,currency_raw,quantity,weight_kg,source_file
0,1,DEC-00497,2024-12-11,import,Turquie,8703.23,27604.85,eur,2148,8433.7,erp_export.xlsx
1,2,DEC-00244,2023-11-15,IMPORT,FRANCE,100199,19262.39,€,3946,62513.9,customs_declarations.csv
2,3,DEC-00552,"Nov 08, 2024",Export,Fr.,6109.10,22070.66,Euro,293,2649.9,customs_declarations.csv
3,4,DEC-00213,08/08/2023,exp,Italy,0804 10,41806.05,EUR,649,3845.5,customs_declarations.csv
4,5,DEC-00549,"May 28, 2024",IMPORT,NaN,3004.90,23209.82,$,2820,21680.6,customs_declarations.csv
5,6,DEC-00506,"Dec 12, 2023",exp,FRANCE,0804.10,17974.83,EUR,586,3396.2,customs_declarations.csv
6,7,DEC-00284,"Aug 05, 2023",Export,FR,080410,15081.15,€,357,559.8,customs_declarations.csv
7,8,DEC-00077,2024-06-13,Exp.,Deutschland,0804-10,17740.15,Euro,3164,48632.3,customs_declarations.csv


## A.2 Agent 1 — Data Profiler  →  `profiler_report.csv`

In [ ]:
ISSUE_ORDER = ["duplicate_declaration", "missing_country", "country_spelling", "hs_format", "hs_invalid",
               "date_format", "flow_label", "currency_label", "negative_value", "mixed_currency"]

def run_profiler():
    raw = pd.read_csv(f"{OUT}/raw_trade_data.csv", dtype=str)
    n = len(raw)
    iso = raw.country_raw.map(lambda s: resolve_country(s)[0])
    digits = raw.hs_code_raw.map(hs_digits)
    fmt = raw.date_raw.map(lambda s: parse_date(s)[1])
    ccy = raw.currency_raw.map(lambda s: CCY_MAP.get(norm_txt(s)))
    val = pd.to_numeric(raw.value_raw)
    checks = [
        ("duplicate_declaration", "declaration_id", raw.duplicated("declaration_id"), "high", "Drop repeated declarations, keep the first occurrence"),
        ("missing_country", "country_raw", raw.country_raw.isna(), "high", "Quarantine rows without a partner country"),
        ("country_spelling", "country_raw", raw.country_raw.notna() & (iso.fillna("") != raw.country_raw.fillna("")), "medium", "Normalise to ISO-3166 alpha-2 codes"),
        ("hs_format", "hs_code_raw", (digits.str.len() == 6) & (digits != raw.hs_code_raw), "low", "Strip punctuation / spaces from HS codes"),
        ("hs_invalid", "hs_code_raw", digits.str.len() != 6, "high", "Quarantine rows with an invalid HS code"),
        ("date_format", "date_raw", fmt != "%Y-%m-%d", "low", "Parse all dates to ISO 8601"),
        ("flow_label", "flow_raw", ~raw.flow_raw.isin(["Export", "Import"]), "low", "Normalise flow labels"),
        ("currency_label", "currency_raw", ~raw.currency_raw.isin(["EUR", "USD", "TND"]), "low", "Normalise currency labels to ISO-4217"),
        ("negative_value", "value_raw", val < 0, "high", "Review negative values with a human"),
        ("mixed_currency", "currency_raw", ccy != "TND", "medium", "Convert all values to TND"),
    ]
    out = []
    for typ, col, mask, sev, action in checks:
        k = int(mask.sum())
        if k == 0:
            continue
        out.append(dict(issue_id=f"P{ISSUE_ORDER.index(typ) + 1:02d}", column=col, issue_type=typ, affected_rows=k,
                        pct_rows=round(100 * k / n, 2), severity=sev, example=str(raw.loc[mask, col].iloc[0]),
                        recommended_action=action, detected_by="Data Profiler", detected_at=ts(20)))
    return pd.DataFrame(out)

profiler_report = run_profiler()
save(profiler_report, "profiler_report.csv")
profiler_report

  saved bi_flow_outputs/profiler_report.csv              10 rows x 10 cols


,issue_id,column,issue_type,affected_rows,pct_rows,severity,example,recommended_action,detected_by,detected_at
0,P01,declaration_id,duplicate_declaration,30,4.76,high,DEC-00539,"Drop repeated declarations, keep the first occurrence",Data Profiler,2026-10-28T09:00:20
1,P02,country_raw,missing_country,18,2.86,high,nan,Quarantine rows without a partner country,Data Profiler,2026-10-28T09:00:20
2,P03,country_raw,country_spelling,490,77.78,medium,Turquie,Normalise to ISO-3166 alpha-2 codes,Data Profiler,2026-10-28T09:00:20
3,P04,hs_code_raw,hs_format,404,64.13,low,8703.23,Strip punctuation / spaces from HS codes,Data Profiler,2026-10-28T09:00:20
4,P05,hs_code_raw,hs_invalid,5,0.79,high,08041,Quarantine rows with an invalid HS code,Data Profiler,2026-10-28T09:00:20
5,P06,date_raw,date_format,324,51.43,low,"Nov 08, 2024",Parse all dates to ISO 8601,Data Profiler,2026-10-28T09:00:20
6,P07,flow_raw,flow_label,512,81.27,low,import,Normalise flow labels,Data Profiler,2026-10-28T09:00:20
7,P08,currency_raw,currency_label,473,75.08,low,eur,Normalise currency labels to ISO-4217,Data Profiler,2026-10-28T09:00:20
8,P09,value_raw,negative_value,5,0.79,high,-30295.62,Review negative values with a human,Data Profiler,2026-10-28T09:00:20
9,P10,currency_raw,mixed_currency,596,94.60,medium,eur,Convert all values to TND,Data Profiler,2026-10-28T09:00:20


## A.3 Agent 2 — Quality & Cleaning  →  `cleaning_log.csv`, `cleaning_rules.csv`, `clean_trade_data.csv`, `quarantine.csv`

Every changed cell is logged with its rule, reason, confidence and human-validation status.

In [ ]:
RULES = {
    "R-DUP-01": ("Drop repeated declaration_id, keep the first occurrence", "deterministic", "P01"),
    "R-CTRY-01": ("Map country spelling to ISO-3166 alpha-2 with the alias reference table", "alias_lookup", "P03"),
    "R-CTRY-02": ("Fuzzy-match a misspelled country to the nearest alias; human validation if confidence < 0.90", "fuzzy_match", "P03"),
    "R-CTRY-03": ("Missing country: set UNK and quarantine the row", "deterministic", "P02"),
    "R-HS-01": ("Strip punctuation and spaces so the HS code has 6 digits", "deterministic", "P04"),
    "R-HS-02": ("HS code is not 6 digits: set INVALID and quarantine the row", "deterministic", "P05"),
    "R-DATE-01": ("Parse the date (YYYY-MM-DD, DD/MM/YYYY or Mon DD, YYYY) into ISO 8601", "deterministic", "P06"),
    "R-FLOW-01": ("Normalise the flow label to Export or Import", "deterministic", "P07"),
    "R-CUR-01": ("Normalise the currency label to its ISO-4217 code", "deterministic", "P08"),
    "R-VAL-01": ("Negative value: use the absolute value and flag it for human review", "heuristic", "P09"),
    "R-FX-01": ("Convert the value to TND with fx_rates.csv", "deterministic", "P10"),
}
QUARANTINE_RULES = {"R-CTRY-03", "R-HS-02"}

def run_cleaning():
    raw = pd.read_csv(f"{OUT}/raw_trade_data.csv", dtype=str)
    fx = rd("fx_rates.csv", num=["rate_to_tnd"]).set_index("currency")["rate_to_tnd"].to_dict()
    events, clean_rows, seen = [], [], set()
    by_row = defaultdict(list)
    counter = [0]

    def log(r, column, before, after, rule, conf, reason):
        counter[0] += 1
        desc, method, issue = RULES[rule]
        needs = rule in QUARANTINE_RULES or rule == "R-VAL-01" or (rule == "R-CTRY-02" and conf < 0.90)
        if not needs:
            status, who = "auto_approved", "system"
        elif rule in QUARANTINE_RULES or rng.random() >= .5:
            status, who = "pending", ""
        else:
            status, who = "approved", "human:karim.trabelsi"
        e = dict(event_id=f"C{counter[0]:04d}", row_id=r.row_id, declaration_id=r.declaration_id, column=column,
                 before=before, after=after, rule_id=rule, rule_description=desc, method=method, confidence=conf,
                 reason=reason, profiler_issue_id=issue, needs_human_validation=needs, approval_status=status,
                 approver=who, timestamp=ts(40 + counter[0] * 0.01))
        events.append(e)
        by_row[r.row_id].append(e)

    for r in raw.itertuples(index=False):
        if r.declaration_id in seen:
            log(r, "row", r.declaration_id, "(row dropped)", "R-DUP-01", 1.0,
                "declaration_id already present in an earlier row of the same file")
            continue
        seen.add(r.declaration_id)

        d, _ = parse_date(r.date_raw)
        if r.date_raw != d.isoformat():
            log(r, "date", r.date_raw, d.isoformat(), "R-DATE-01", 1.0, "date written in a non-ISO format")

        flow = FLOW_MAP[norm_txt(r.flow_raw)]
        if r.flow_raw != flow:
            log(r, "flow", r.flow_raw, flow, "R-FLOW-01", 1.0, "flow label written in a non-standard way")

        iso, how, conf = resolve_country(r.country_raw)
        if how == "missing" or iso is None:
            iso = "UNK"
            log(r, "country", "(missing)", "UNK", "R-CTRY-03", 0.0, "no partner country in the declaration")
        elif r.country_raw != iso:
            rule = "R-CTRY-01" if how == "alias_lookup" else "R-CTRY-02"
            why = ("country name found in the alias reference table" if how == "alias_lookup"
                   else f"misspelled country name matched to the nearest alias with similarity {conf}")
            log(r, "country", r.country_raw, iso, rule, conf, why)

        digits = hs_digits(r.hs_code_raw)
        if len(digits) != 6:
            log(r, "hs_code", r.hs_code_raw, "INVALID", "R-HS-02", 0.0, f"HS code has {len(digits)} digits instead of 6")
            hs6 = "INVALID"
        else:
            hs6 = digits
            if r.hs_code_raw != digits:
                log(r, "hs_code", r.hs_code_raw, digits, "R-HS-01", 1.0, "HS code contained punctuation or spaces")

        v = float(r.value_raw)
        if v < 0:
            log(r, "value", f"{v:.2f}", f"{abs(v):.2f}", "R-VAL-01", 0.60,
                "negative trade value is not plausible; absolute value used pending human review")
            v = abs(v)
        c = CCY_MAP[norm_txt(r.currency_raw)]
        if r.currency_raw != c:
            log(r, "currency", r.currency_raw, c, "R-CUR-01", 1.0, "currency label was not an ISO-4217 code")
        value_tnd = round(v * fx[c], 2)
        log(r, "value_tnd", f"{v:.2f} {c}", f"{value_tnd:.2f} TND", "R-FX-01", 1.0,
            f"converted at {fx[c]} TND per {c} (fx_rates.csv)")

        clean_rows.append(dict(row_id=r.row_id, declaration_id=r.declaration_id, date=d.isoformat(), flow=flow,
                               country_iso=iso, hs6=hs6, value_tnd=value_tnd, quantity=r.quantity,
                               weight_kg=r.weight_kg, source_file=r.source_file))
    clean = pd.DataFrame(clean_rows)

    def flag(row):
        if row.country_iso == "UNK" or row.hs6 == "INVALID":
            return "quarantined"
        if any(e["approval_status"] == "pending" for e in by_row[row.row_id]):
            return "pending_review"
        return "ok"
    clean["quality_flag"] = clean.apply(flag, axis=1)
    return pd.DataFrame(events), clean

cleaning_log, clean_df = run_cleaning()
save(cleaning_log, "cleaning_log.csv")
save(pd.DataFrame([dict(rule_id=k, description=v[0], method=v[1], profiler_issue_id=v[2]) for k, v in RULES.items()]),
     "cleaning_rules.csv")
save(clean_df, "clean_trade_data.csv")
save(clean_df[clean_df.quality_flag == "quarantined"], "quarantine.csv")
print("\nrows by quality flag:", clean_df.quality_flag.value_counts().to_dict())
cleaning_log[cleaning_log.rule_id != "R-FX-01"].head(8)

  saved bi_flow_outputs/cleaning_log.csv               2752 rows x 16 cols
  saved bi_flow_outputs/cleaning_rules.csv               11 rows x  4 cols
  saved bi_flow_outputs/clean_trade_data.csv            600 rows x 11 cols
  saved bi_flow_outputs/quarantine.csv                   21 rows x 11 cols

rows by quality flag: {'ok': 568, 'quarantined': 21, 'pending_review': 11}


,event_id,row_id,declaration_id,column,before,after,rule_id,rule_description,method,confidence,reason,profiler_issue_id,needs_human_validation,approval_status,approver,timestamp
0,C0001,1,DEC-00497,flow,import,Import,R-FLOW-01,Normalise the flow label to Export or Import,deterministic,1.0,flow label written in a non-standard way,P07,False,auto_approved,system,2026-10-28T09:00:40
1,C0002,1,DEC-00497,country,Turquie,TR,R-CTRY-01,Map country spelling to ISO-3166 alpha-2 with the alias reference table,alias_lookup,1.0,country name found in the alias reference table,P03,False,auto_approved,system,2026-10-28T09:00:40
2,C0003,1,DEC-00497,hs_code,8703.23,870323,R-HS-01,Strip punctuation and spaces so the HS code has 6 digits,deterministic,1.0,HS code contained punctuation or spaces,P04,False,auto_approved,system,2026-10-28T09:00:40
3,C0004,1,DEC-00497,currency,eur,EUR,R-CUR-01,Normalise the currency label to its ISO-4217 code,deterministic,1.0,currency label was not an ISO-4217 code,P08,False,auto_approved,system,2026-10-28T09:00:40
5,C0006,2,DEC-00244,flow,IMPORT,Import,R-FLOW-01,Normalise the flow label to Export or Import,deterministic,1.0,flow label written in a non-standard way,P07,False,auto_approved,system,2026-10-28T09:00:40
6,C0007,2,DEC-00244,country,FRANCE,FR,R-CTRY-01,Map country spelling to ISO-3166 alpha-2 with the alias reference table,alias_lookup,1.0,country name found in the alias reference table,P03,False,auto_approved,system,2026-10-28T09:00:40
7,C0008,2,DEC-00244,currency,€,EUR,R-CUR-01,Normalise the currency label to its ISO-4217 code,deterministic,1.0,currency label was not an ISO-4217 code,P08,False,auto_approved,system,2026-10-28T09:00:40
9,C0010,3,DEC-00552,date,"Nov 08, 2024",2024-11-08,R-DATE-01,"Parse the date (YYYY-MM-DD, DD/MM/YYYY or Mon DD, YYYY) into ISO 8601",deterministic,1.0,date written in a non-ISO format,P06,False,auto_approved,system,2026-10-28T09:00:40


## A.4 Agent 3 — ETL / Data Engineering  →  star schema CSVs, `etl_log.csv`, `column_lineage.csv`

In [ ]:
def run_etl():
    c = rd("clean_trade_data.csv", num=["value_tnd", "quantity", "weight_kg"])
    steps = []

    def step(no, op, inp, outp, rin, rout, cols, desc):
        steps.append(dict(step_id=f"E{no:02d}", step_no=no, operation=op, input_tables=inp, output_table=outp,
                          rows_in=rin, rows_out=rout, columns_used=cols, description=desc, timestamp=ts(60 + no)))

    ok = c[c.quality_flag != "quarantined"].copy()
    step(1, "filter", "clean_trade_data", "clean_trade_data (valid)", len(c), len(ok), "quality_flag",
         "Exclude quarantined rows (missing country or invalid HS code) from the analytical model.")

    dc = ok[["country_iso"]].drop_duplicates().sort_values("country_iso").reset_index(drop=True)
    dc.insert(0, "country_key", range(1, len(dc) + 1))
    dc["country_name"] = dc.country_iso.map(lambda x: COUNTRIES[x][0])
    dc["region"] = dc.country_iso.map(lambda x: COUNTRIES[x][1])
    step(2, "build_dimension", "clean_trade_data (valid)", "Dim_Country", len(ok), len(dc), "country_iso",
         "Build Dim_Country with one surrogate key per ISO country code.")

    dp = ok[["hs6"]].drop_duplicates().sort_values("hs6").reset_index(drop=True)
    dp.insert(0, "product_key", range(1, len(dp) + 1))
    dp["description"] = dp.hs6.map(PRODUCTS)
    dp["hs_chapter"] = dp.hs6.str[:2]
    step(3, "build_dimension", "clean_trade_data (valid)", "Dim_Product", len(ok), len(dp), "hs6",
         "Build Dim_Product with one surrogate key per 6-digit HS code and its description.")

    dd = ok[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
    dts = pd.to_datetime(dd["date"])
    dd.insert(0, "date_key", dts.dt.strftime("%Y%m%d").astype(int))
    dd["year"], dd["month"], dd["quarter"] = dts.dt.year, dts.dt.month, dts.dt.quarter
    step(4, "build_dimension", "clean_trade_data (valid)", "Dim_Date", len(ok), len(dd), "date",
         "Build Dim_Date (date_key = YYYYMMDD) with year, month and quarter.")

    f = ok.copy()
    f["date_key"] = pd.to_datetime(f["date"]).dt.strftime("%Y%m%d").astype(int)
    f = f.merge(dc[["country_key", "country_iso"]], on="country_iso").merge(dp[["product_key", "hs6"]], on="hs6")
    f.insert(0, "trade_id", range(1, len(f) + 1))
    f = f[["trade_id", "row_id", "date_key", "country_key", "product_key", "flow", "value_tnd", "quantity", "weight_kg"]]
    step(5, "build_fact", "clean_trade_data (valid) + dimensions", "Fact_Trade", len(ok), len(f),
         "value_tnd|flow|quantity|weight_kg|date_key|country_key|product_key",
         "Build Fact_Trade by joining the cleaned rows to the three dimensions on surrogate keys.")

    assert len(f) == len(ok), "referential integrity broken"
    step(6, "validate", "Fact_Trade", "Fact_Trade", len(f), len(f), "all keys",
         "Referential integrity check: every fact row found its country, product and date key (0 orphan rows).")
    etl = pd.DataFrame(steps)

    base_rules = ["R-DUP-01", "R-CTRY-03", "R-HS-02"]       # these rules decide which rows exist at all
    def cl(col, src, raw_cols, how, e, rules):
        return dict(fact_column=col, source_table="clean_trade_data", source_column=src, raw_columns=raw_cols,
                    transformation=how, etl_step="|".join(["E01"] + e + ["E06"]),
                    cleaning_rules="|".join(base_rules + [r for r in rules if r not in base_rules]))
    col_lineage = pd.DataFrame([
        cl("Fact_Trade[value_tnd]", "value_tnd", "value_raw|currency_raw", "value converted to TND", ["E05"], ["R-CUR-01", "R-VAL-01", "R-FX-01"]),
        cl("Fact_Trade[flow]", "flow", "flow_raw", "copied", ["E05"], ["R-FLOW-01"]),
        cl("Fact_Trade[quantity]", "quantity", "quantity", "copied", ["E05"], []),
        cl("Fact_Trade[weight_kg]", "weight_kg", "weight_kg", "copied", ["E05"], []),
        cl("Fact_Trade[date_key]", "date", "date_raw", "YYYYMMDD surrogate key", ["E04", "E05"], ["R-DATE-01"]),
        cl("Fact_Trade[country_key]", "country_iso", "country_raw", "surrogate key lookup", ["E02", "E05"], ["R-CTRY-01", "R-CTRY-02"]),
        cl("Fact_Trade[product_key]", "hs6", "hs_code_raw", "surrogate key lookup", ["E03", "E05"], ["R-HS-01"]),
        cl("Dim_Date[year]", "date", "date_raw", "year extracted", ["E04"], ["R-DATE-01"]),
        cl("Dim_Date[month]", "date", "date_raw", "month extracted", ["E04"], ["R-DATE-01"]),
        cl("Dim_Country[country_iso]", "country_iso", "country_raw", "distinct ISO code", ["E02"], ["R-CTRY-01", "R-CTRY-02"]),
        cl("Dim_Product[hs6]", "hs6", "hs_code_raw", "distinct 6-digit HS code", ["E03"], ["R-HS-01"]),
    ])
    return dc, dp, dd, f, etl, col_lineage

dim_country, dim_product, dim_date, fact_trade, etl_log, column_lineage = run_etl()
save(dim_country, "dim_country.csv"); save(dim_product, "dim_product.csv"); save(dim_date, "dim_date.csv")
save(fact_trade, "fact_trade.csv"); save(etl_log, "etl_log.csv"); save(column_lineage, "column_lineage.csv")
etl_log[["step_id", "operation", "output_table", "rows_in", "rows_out", "description"]]

  saved bi_flow_outputs/dim_country.csv                  13 rows x  4 cols
  saved bi_flow_outputs/dim_product.csv                   9 rows x  4 cols
  saved bi_flow_outputs/dim_date.csv                    374 rows x  5 cols
  saved bi_flow_outputs/fact_trade.csv                  579 rows x  9 cols
  saved bi_flow_outputs/etl_log.csv                       6 rows x 10 cols
  saved bi_flow_outputs/column_lineage.csv               11 rows x  7 cols


,step_id,operation,output_table,rows_in,rows_out,description
0,E01,filter,clean_trade_data (valid),600,579,Exclude quarantined rows (missing country or invalid HS code) from the analytical model.
1,E02,build_dimension,Dim_Country,579,13,Build Dim_Country with one surrogate key per ISO country code.
2,E03,build_dimension,Dim_Product,579,9,Build Dim_Product with one surrogate key per 6-digit HS code and its description.
3,E04,build_dimension,Dim_Date,579,374,"Build Dim_Date (date_key = YYYYMMDD) with year, month and quarter."
4,E05,build_fact,Fact_Trade,579,579,Build Fact_Trade by joining the cleaned rows to the three dimensions on surrogate keys.
5,E06,validate,Fact_Trade,579,579,"Referential integrity check: every fact row found its country, product and date key (0..."


## A.5 Agent 4 — Semantic & KPI  →  `kpi_catalog.csv`, `kpi_values.csv`

In [ ]:
def load_model():
    f = rd("fact_trade.csv", num=["trade_id", "date_key", "country_key", "product_key", "value_tnd", "quantity", "weight_kg"])
    dd = rd("dim_date.csv", num=["date_key", "year", "month", "quarter"])
    dc = rd("dim_country.csv", num=["country_key"])
    dp = rd("dim_product.csv", num=["product_key"])
    return f.merge(dd, on="date_key").merge(dc, on="country_key").merge(dp, on="product_key")

KPI_DEFS = [
    ("K01", "Total Exports", 'CALCULATE(SUM(Fact_Trade[value_tnd]), Fact_Trade[flow] = "Export")',
     "Fact_Trade[value_tnd]|Fact_Trade[flow]", "TND", "Sum of export values in TND.", "BO3"),
    ("K02", "Total Imports", 'CALCULATE(SUM(Fact_Trade[value_tnd]), Fact_Trade[flow] = "Import")',
     "Fact_Trade[value_tnd]|Fact_Trade[flow]", "TND", "Sum of import values in TND.", "BO3"),
    ("K03", "Trade Balance", "[Total Exports] - [Total Imports]",
     "Fact_Trade[value_tnd]|Fact_Trade[flow]", "TND", "Exports minus imports; a positive value is a trade surplus.", "BO3"),
    ("K04", "Coverage Ratio", "DIVIDE([Total Exports], [Total Imports])",
     "Fact_Trade[value_tnd]|Fact_Trade[flow]", "ratio", "Exports divided by imports; above 1 means exports cover imports.", "BO3"),
    ("K05", "Export YoY Growth",
     "VAR cur = [Total Exports] VAR prev = CALCULATE([Total Exports], SAMEPERIODLASTYEAR(Dim_Date[date])) RETURN DIVIDE(cur - prev, prev)",
     "Fact_Trade[value_tnd]|Fact_Trade[flow]|Dim_Date[year]", "percent", "Year-over-year growth of exports.", "BO3"),
]

def run_kpi():
    F = load_model()
    cat = pd.DataFrame(KPI_DEFS, columns=["kpi_id", "name", "dax", "fact_columns", "unit", "description", "business_objective"])
    vals = []

    def add(kid, st, sv, period, v, n):
        vals.append(dict(kpi_id=kid, scope_type=st, scope_value=sv, period=str(period), value=float(v),
                         rows_used=int(n), computed_at=ts(80)))
    for y in (2023, 2024):
        s = F[F.year == y]
        e, i = s[s.flow == "Export"].value_tnd.sum(), s[s.flow == "Import"].value_tnd.sum()
        add("K01", "all", "All", y, e, (s.flow == "Export").sum())
        add("K02", "all", "All", y, i, (s.flow == "Import").sum())
        add("K03", "all", "All", y, e - i, len(s))
        add("K04", "all", "All", y, e / i, len(s))
        for iso, g in s.groupby("country_iso"):
            ge, gi = g[g.flow == "Export"], g[g.flow == "Import"]
            if len(ge): add("K01", "country", iso, y, ge.value_tnd.sum(), len(ge))
            if len(gi): add("K02", "country", iso, y, gi.value_tnd.sum(), len(gi))
    e23 = F[(F.year == 2023) & (F.flow == "Export")].value_tnd.sum()
    e24 = F[(F.year == 2024) & (F.flow == "Export")].value_tnd.sum()
    add("K05", "all", "All", 2024, (e24 - e23) / e23, ((F.flow == "Export")).sum())
    return cat, pd.DataFrame(vals)

kpi_catalog, kpi_values = run_kpi()
save(kpi_catalog, "kpi_catalog.csv"); save(kpi_values, "kpi_values.csv")
kpi_values[kpi_values.scope_type == "all"]

  saved bi_flow_outputs/kpi_catalog.csv                   5 rows x  7 cols
  saved bi_flow_outputs/kpi_values.csv                   47 rows x  7 cols


,kpi_id,scope_type,scope_value,period,value,rows_used,computed_at
0,K01,all,All,2023,1.899046e+07,147,2026-10-28T09:01:20
1,K02,all,All,2023,1.800000e+07,125,2026-10-28T09:01:20
2,K03,all,All,2023,9.904632e+05,272,2026-10-28T09:01:20
3,K04,all,All,2023,1.055026e+00,272,2026-10-28T09:01:20
23,K01,all,All,2024,2.225161e+07,173,2026-10-28T09:01:20
24,K02,all,All,2024,2.167990e+07,134,2026-10-28T09:01:20
25,K03,all,All,2024,5.717115e+05,307,2026-10-28T09:01:20
26,K04,all,All,2024,1.026371e+00,307,2026-10-28T09:01:20
46,K05,all,All,2024,1.717258e-01,320,2026-10-28T09:01:20


## A.6 Agents 5-7 — BI Analyst, Dashboard Generator, Orchestrator  →  `analyst_insights.csv`, `dashboard_visuals.csv`, `orchestrator_runs.csv`

In [ ]:
def run_analyst():
    F = load_model()
    exp, imp = F[F.flow == "Export"], F[F.flow == "Import"]
    out = []

    def add(iid, kid, text, method, flow, years, expr, group_by, rows, mname, mval, thr, rule, trig, rec_yes, rec_no):
        out.append(dict(insight_id=iid, kpi_id=kid, text=text, method=method, flow=flow, years=years, filter_expr=expr,
                        group_by=group_by, rows_used=int(rows), metric_name=mname, metric_value=round(float(mval), 2),
                        threshold=thr, rule_text=rule, rule_triggered=bool(trig),
                        recommendation=rec_yes if trig else rec_no, created_at=ts(90)))

    # I01 - largest export decline by partner country
    a = exp.pivot_table(index="country_iso", columns="year", values="value_tnd", aggfunc="sum").fillna(0)
    a = a[a[2023] >= a[2023].median()]
    g = (a[2024] - a[2023]) / a[2023] * 100
    iso = g.idxmin()
    n = len(exp[(exp.country_iso == iso)])
    add("I01", "K01",
        f"Exports to {COUNTRIES[iso][0]} fell {abs(g[iso]):.1f}% in 2024 versus 2023 ({a.loc[iso, 2023]:,.0f} to {a.loc[iso, 2024]:,.0f} TND).",
        "yoy_by_country", "Export", "2023|2024", f"country_iso == '{iso}'", "country_iso", n,
        "export_yoy_pct", g[iso], -10.0, "Recommend action when a major partner's export YoY change is below -10%.", g[iso] < -10,
        f"Investigate the commercial relationship with {COUNTRIES[iso][0]} and diversify towards growing markets.",
        "No action required; keep monitoring partner trends.")

    # I02 - export product concentration
    p = exp[exp.year == 2024].groupby("hs6").value_tnd.sum()
    hs = p.idxmax(); share = p[hs] / p.sum() * 100
    add("I02", "K01",
        f"{PRODUCTS[hs]} (HS {hs}) is the top export product of 2024 with {share:.1f}% of export value.",
        "top_share_by_product", "Export", "2024", f"hs6 == '{hs}'", "hs6", len(exp[(exp.year == 2024) & (exp.hs6 == hs)]),
        "top_product_share_pct", share, 30.0, "Flag concentration risk when one product exceeds 30% of export value.", share >= 30,
        f"Reduce dependence on {PRODUCTS[hs]} by supporting other export lines.",
        "No concentration risk detected on products; keep monitoring.")

    # I03 - monthly anomaly (z-score)
    m = exp.groupby(["year", "month"]).value_tnd.sum()
    z = (m - m.mean()) / m.std()
    (yy, mm) = z.abs().idxmax()
    add("I03", "K01",
        f"Exports in {yy}-{mm:02d} reached {m[(yy, mm)]:,.0f} TND, a z-score of {z[(yy, mm)]:.2f} against the monthly average.",
        "monthly_zscore", "Export", f"{yy}", f"year == {yy} and month == {mm}", "year,month",
        len(exp[(exp.year == yy) & (exp.month == mm)]), "zscore", z[(yy, mm)], 2.0,
        "Flag a month as an anomaly when its absolute z-score is at least 2.", abs(z[(yy, mm)]) >= 2,
        f"Check the large declarations of {yy}-{mm:02d} for one-off shipments or data errors before reporting.",
        "No monthly anomaly detected.")

    # I04 - import supplier concentration
    s = imp[imp.year == 2024].groupby("country_iso").value_tnd.sum()
    top = s.idxmax(); sh = s[top] / s.sum() * 100
    add("I04", "K02",
        f"{COUNTRIES[top][0]} is the top import supplier of 2024 with {sh:.1f}% of import value.",
        "top_share_by_supplier", "Import", "2024", f"country_iso == '{top}'", "country_iso",
        len(imp[(imp.year == 2024) & (imp.country_iso == top)]), "top_supplier_share_pct", sh, 25.0,
        "Flag supplier concentration when one country exceeds 25% of import value.", sh >= 25,
        f"Diversify import sourcing away from {COUNTRIES[top][0]}.",
        "No supplier concentration risk detected.")
    return pd.DataFrame(out)

analyst_insights = run_analyst()
save(analyst_insights, "analyst_insights.csv")

dashboard_visuals = pd.DataFrame([
    ("V01", "Overview", "card", "Total Exports", "K01", "none", "", "A single headline number is best read as a card."),
    ("V02", "Overview", "card", "Total Imports", "K02", "none", "", "A single headline number is best read as a card."),
    ("V03", "Overview", "card", "Trade Balance", "K03", "none", "", "A single headline number is best read as a card."),
    ("V04", "Overview", "gauge", "Coverage Ratio", "K04", "none", "", "A ratio with a natural target of 1 suits a gauge."),
    ("V05", "Overview", "card", "Export YoY Growth", "K05", "none", "", "A growth rate is read as one signed number."),
    ("V06", "Markets", "bar", "Exports by partner country", "K01", "country", "I01", "Ranking categories by value suits a bar chart."),
    ("V07", "Products", "donut", "Exports by product", "K01", "product", "I02", "Parts of a whole suit a donut chart."),
    ("V08", "Trends", "line", "Monthly exports", "K01", "month", "I03", "A time series suits a line chart."),
    ("V09", "Markets", "bar", "Imports by supplier country", "K02", "country", "I04", "Ranking categories by value suits a bar chart."),
], columns=["visual_id", "page", "visual_type", "title", "kpi_ids", "dimension", "insight_id", "chosen_because"])
dashboard_visuals["generated_at"] = ts(100)
save(dashboard_visuals, "dashboard_visuals.csv")

n_h = int(cleaning_log.needs_human_validation.sum())
n_a = int(((cleaning_log.needs_human_validation) & (cleaning_log.approval_status == "approved")).sum())
steps = [
    ("Data Profiler", "success", 0, "", "no", "", "profiler_report.csv", 20),
    ("Quality & Cleaning", "success", 0, "", "yes", f"{n_a} of {n_h} approved, {n_h - n_a} pending",
     "cleaning_log.csv; clean_trade_data.csv; quarantine.csv", 40),
    ("ETL / Data Engineering", "success", 1, "TimeoutError while building Dim_Date; retry succeeded", "no", "",
     "dim_*.csv; fact_trade.csv; etl_log.csv; column_lineage.csv", 60),
    ("Semantic & KPI", "success", 0, "", "no", "", "kpi_catalog.csv; kpi_values.csv", 80),
    ("BI Analyst", "success", 0, "", "no", "", "analyst_insights.csv", 90),
    ("Dashboard Generator", "success", 0, "", "no", "", "dashboard_visuals.csv", 100),
]
orchestrator_runs = pd.DataFrame([dict(
    run_id="RUN-2026-10-28-001", step_no=i + 1, agent=a, status=s, started_at=ts(t), ended_at=ts(t + 12 + 5 * r),
    retries=r, error=err, human_validation_required=hv, human_validation_status=hvs, outputs=o)
    for i, (a, s, r, err, hv, hvs, o, t) in enumerate(steps)])
save(orchestrator_runs, "orchestrator_runs.csv")
analyst_insights[["insight_id", "text", "metric_value", "threshold", "rule_triggered"]]

  saved bi_flow_outputs/analyst_insights.csv              4 rows x 16 cols
  saved bi_flow_outputs/dashboard_visuals.csv             9 rows x  9 cols
  saved bi_flow_outputs/orchestrator_runs.csv             6 rows x 11 cols


,insight_id,text,metric_value,threshold,rule_triggered
0,I01,"Exports to Spain fell 17.4% in 2024 versus 2023 (2,177,544 to 1,799,579 TND).",-17.36,-10.0,True
1,I02,Virgin olive oil (HS 150910) is the top export product of 2024 with 40.0% of export va...,39.99,30.0,True
2,I03,"Exports in 2024-07 reached 3,608,249 TND, a z-score of 2.91 against the monthly average.",2.91,2.0,True
3,I04,China is the top import supplier of 2024 with 27.4% of import value.,27.41,25.0,True


In [ ]:
print(f"\n{len(os.listdir(OUT))} files in {OUT}/ :")
for fn in sorted(os.listdir(OUT)):
    if fn.endswith(".csv"):
        print("  -", fn)


18 files in bi_flow_outputs/ :
  - analyst_insights.csv
  - clean_trade_data.csv
  - cleaning_log.csv
  - cleaning_rules.csv
  - column_lineage.csv
  - dashboard_visuals.csv
  - dim_country.csv
  - dim_date.csv
  - dim_product.csv
  - etl_log.csv
  - fact_trade.csv
  - fx_rates.csv
  - kpi_catalog.csv
  - kpi_values.csv
  - orchestrator_runs.csv
  - profiler_report.csv
  - quarantine.csv
  - raw_trade_data.csv


---
# PART B — The BI Auditor / XAI agent

From here on the Auditor reads **only the CSV files**. To audit your real pipeline, replace the files in `bi_flow_outputs/` (same column names).

## B.1 Load every agent output

In [ ]:
prof = rd("profiler_report.csv", num=["affected_rows", "pct_rows"]).fillna("")
rules_df = rd("cleaning_rules.csv").fillna("")
clog = rd("cleaning_log.csv", num=["confidence"]).fillna("")
clog["needs_human_validation"] = clog.needs_human_validation == "True"
clean = rd("clean_trade_data.csv", num=["value_tnd", "quantity", "weight_kg"]).fillna("")
clean["year"] = clean.date.str[:4].astype(int)
clean["month"] = clean.date.str[5:7].astype(int)
etl = rd("etl_log.csv", num=["step_no", "rows_in", "rows_out"]).fillna("")
collin = rd("column_lineage.csv").fillna("")
kpi_cat = rd("kpi_catalog.csv").fillna("")
kpi_val = rd("kpi_values.csv", num=["value", "rows_used"]).fillna("")
ins = rd("analyst_insights.csv", num=["metric_value", "threshold", "rows_used"]).fillna("")
ins["rule_triggered"] = ins.rule_triggered == "True"
vis = rd("dashboard_visuals.csv").fillna("")
orch = rd("orchestrator_runs.csv", num=["step_no", "retries"]).fillna("")
raw = rd("raw_trade_data.csv")

print("Loaded:", {k: len(v) for k, v in dict(profiler=prof, rules=rules_df, cleaning_log=clog, clean=clean, etl=etl,
      kpi_catalog=kpi_cat, kpi_values=kpi_val, insights=ins, visuals=vis, orchestrator=orch).items()})

Loaded: {'profiler': 10, 'rules': 11, 'cleaning_log': 2752, 'clean': 600, 'etl': 6, 'kpi_catalog': 5, 'kpi_values': 47, 'insights': 4, 'visuals': 9, 'orchestrator': 6}


## B.2 Unified audit trail (`audit_trail.csv`)

Every agent action becomes one event with an id, a text summary and a timestamp. These summaries are what the RAG will index.

In [ ]:
def fmt_kpi(kid, v):
    if kid == "K04": return f"{v:.4f}"
    if kid == "K05": return f"{v * 100:.2f}%"
    return f"{v:,.0f} TND"

events = []
def ev(eid, agent, etype, subject, summary, when):
    events.append(dict(event_id=eid, agent=agent, event_type=etype, subject=subject, summary=summary, timestamp=when))

for r in prof.itertuples():
    ev(r.issue_id, "Data Profiler", "issue_detected", r.column,
       f"Data Profiler detected {r.issue_type} in column {r.column}: {r.affected_rows} rows ({r.pct_rows}%), severity {r.severity}. "
       f"Example: {r.example}. Recommended action: {r.recommended_action}.", r.detected_at)
for r in rules_df.itertuples():
    ev(r.rule_id, "Quality & Cleaning", "rule_definition", r.rule_id,
       f"Cleaning rule {r.rule_id}: {r.description}. Method: {r.method}. Addresses profiler issue {r.profiler_issue_id}.", "")
for r in clog.itertuples():
    who = f" by {r.approver}" if r.approver else ""
    ev(r.event_id, "Quality & Cleaning", "correction", f"row {r.row_id}",
       f"Row {r.row_id} ({r.declaration_id}): rule {r.rule_id} changed {r.column} from '{r.before}' to '{r.after}'. "
       f"Reason: {r.reason}. Confidence {r.confidence}. Human validation required: "
       f"{'yes' if r.needs_human_validation else 'no'}; status {r.approval_status}{who}.", r.timestamp)
for r in etl.itertuples():
    ev(r.step_id, "ETL / Data Engineering", "etl_step", r.output_table,
       f"ETL step {r.step_no} ({r.operation}): {r.description} Input: {r.input_tables}. Output: {r.output_table}. "
       f"Rows {r.rows_in} to {r.rows_out}. Columns: {r.columns_used}.", r.timestamp)
for r in kpi_cat.itertuples():
    ev(r.kpi_id, "Semantic & KPI", "kpi_definition", r.name,
       f"KPI {r.kpi_id} {r.name} is defined in DAX as {r.dax}. It depends on {r.fact_columns}. {r.description}", "")
for r in ins.itertuples():
    ev(r.insight_id, "BI Analyst", "insight", r.kpi_id,
       f"Insight {r.insight_id} (KPI {r.kpi_id}): {r.text} Method {r.method}, flow {r.flow}, years {r.years}, "
       f"filter [{r.filter_expr}], {r.rows_used} rows. Metric {r.metric_name} = {r.metric_value}. "
       f"Rule: {r.rule_text} Threshold {r.threshold}. Rule triggered: {'yes' if r.rule_triggered else 'no'}. "
       f"Recommendation: {r.recommendation}", r.created_at)
for r in vis.itertuples():
    ev(r.visual_id, "Dashboard Generator", "visual", r.title,
       f"Visual {r.visual_id} ({r.visual_type}) '{r.title}' on page {r.page} shows {r.kpi_ids} by {r.dimension}. "
       f"Chosen because: {r.chosen_because}", r.generated_at)
for r in orch.itertuples():
    ev(f"O{r.step_no:02d}", "Orchestrator", "run_step", r.agent,
       f"Run {r.run_id} step {r.step_no}: agent {r.agent} finished with status {r.status} after {r.retries} retries"
       f"{(' (' + r.error + ')') if r.error else ''}. Human validation required: {r.human_validation_required}"
       f"{(' - ' + r.human_validation_status) if r.human_validation_status else ''}. Outputs: {r.outputs}.", r.started_at)

trail = pd.DataFrame(events)
save(trail, "audit_trail.csv")
TRAIL = trail.set_index("event_id")["summary"].to_dict()
print(trail.groupby("agent").size())

  saved bi_flow_outputs/audit_trail.csv                2803 rows x  6 cols
agent
BI Analyst                   4
Dashboard Generator          9
Data Profiler               10
ETL / Data Engineering       6
Orchestrator                 6
Quality & Cleaning        2763
Semantic & KPI               5
dtype: int64


## B.3 Lineage, provenance and independent KPI reconciliation

* **Lineage:** visual → KPI → fact columns → ETL steps → cleaning rules → profiler issues → raw columns and files.
* **Provenance:** how much of a figure comes from rows that were corrected, and how many rows are still pending human validation.
* **Reconciliation:** the Auditor recomputes each published KPI from `clean_trade_data.csv` with its own code and compares.

In [ ]:
def split(s): return [x for x in str(s).split("|") if x]

DIM_COLS = {"none": [], "country": ["Fact_Trade[country_key]", "Dim_Country[country_iso]"],
            "product": ["Fact_Trade[product_key]", "Dim_Product[hs6]"], "month": ["Fact_Trade[date_key]", "Dim_Date[year]", "Dim_Date[month]"]}
RULE_ISSUE = rules_df.set_index("rule_id")["profiler_issue_id"].to_dict()

def lineage_for(kpi_id, dimension="none"):
    k = kpi_cat[kpi_cat.kpi_id == kpi_id].iloc[0]
    cols = split(k.fact_columns) + DIM_COLS[dimension]
    cl = collin[collin.fact_column.isin(cols)]
    rules = sorted({r for s in cl.cleaning_rules for r in split(s)})
    return dict(kpi_id=kpi_id, kpi_name=k["name"], fact_columns="|".join(cols),
                etl_steps="|".join(sorted({e for s in cl.etl_step for e in split(s)})),
                cleaning_rules="|".join(rules),
                profiler_issues="|".join(sorted({RULE_ISSUE[r] for r in rules})),
                raw_columns="|".join(sorted({c for s in cl.raw_columns for c in split(s)})),
                raw_sources="|".join(sorted(raw.source_file.unique())))

lin_rows = []
for v in vis.itertuples():
    for kid in split(v.kpi_ids.replace(",", "|")):
        lin_rows.append(dict(visual_id=v.visual_id, visual_title=v.title, **lineage_for(kid, v.dimension)))
lineage = pd.DataFrame(lin_rows)
save(lineage, "lineage.csv")

BASE = clean[clean.quality_flag != "quarantined"]

def prov_stats(sub):
    ids = set(sub.row_id)
    e = clog[clog.row_id.isin(ids)]
    norm = set(e[e.rule_id != "R-FX-01"].row_id)                         # any format normalisation
    judg = set(e[e.needs_human_validation].row_id)                       # fuzzy match / negative value: needs judgment
    pend = set(e[e.needs_human_validation & (e.approval_status == "pending")].row_id)
    tot = sub.value_tnd.sum()
    return dict(rows=len(sub), normalised_rows=len(norm), judgment_rows=len(judg), pending_rows=len(pend),
                judgment_value_pct=round(100 * sub[sub.row_id.isin(judg)].value_tnd.sum() / tot, 1) if tot else 0.0,
                top_rules=e[e.rule_id != "R-FX-01"].rule_id.value_counts().head(3).to_dict())

def subset_for(kpi_id, year, country=None):
    s = BASE[BASE.year == year] if kpi_id != "K05" else BASE[BASE.year.isin([year - 1, year])]
    if kpi_id in ("K01", "K05"): s = s[s.flow == "Export"]
    if kpi_id == "K02": s = s[s.flow == "Import"]
    if country: s = s[s.country_iso == country]
    return s

def recompute(kpi_id, scope_type, scope_value, period):
    y = int(period); c = scope_value if scope_type == "country" else None
    def tot(flow, yr):
        s = BASE[(BASE.year == yr) & (BASE.flow == flow)]
        if c: s = s[s.country_iso == c]
        return s.value_tnd.sum()
    if kpi_id == "K01": return tot("Export", y)
    if kpi_id == "K02": return tot("Import", y)
    if kpi_id == "K03": return tot("Export", y) - tot("Import", y)
    if kpi_id == "K04": return tot("Export", y) / tot("Import", y)
    if kpi_id == "K05": return (tot("Export", y) - tot("Export", y - 1)) / tot("Export", y - 1)

rec = []
for r in kpi_val.itertuples():
    mine = recompute(r.kpi_id, r.scope_type, r.scope_value, r.period)
    rec.append(dict(kpi_id=r.kpi_id, scope_type=r.scope_type, scope_value=r.scope_value, period=r.period,
                    published=r.value, recomputed=mine, abs_diff=abs(mine - r.value),
                    status="MATCH" if abs(mine - r.value) <= 1e-6 * max(1, abs(r.value)) else "MISMATCH"))
for r in ins.itertuples():
    n = len(BASE.query(f"flow == '{r.flow}' and year in {[int(y) for y in split(r.years)]}" + (f" and {r.filter_expr}" if r.filter_expr else "")))
    rec.append(dict(kpi_id=r.insight_id, scope_type="insight_rows", scope_value=r.filter_expr, period=r.years,
                    published=r.rows_used, recomputed=n, abs_diff=abs(n - r.rows_used),
                    status="MATCH" if n == r.rows_used else "MISMATCH"))
recon = pd.DataFrame(rec)
save(recon, "reconciliation.csv")
print(recon.status.value_counts().to_dict())
lineage.head(3)

  saved bi_flow_outputs/lineage.csv                       9 rows x 10 cols
  saved bi_flow_outputs/reconciliation.csv               51 rows x  8 cols
{'MATCH': 51}


,visual_id,visual_title,kpi_id,kpi_name,fact_columns,etl_steps,cleaning_rules,profiler_issues,raw_columns,raw_sources
0,V01,Total Exports,K01,Total Exports,Fact_Trade[value_tnd]|Fact_Trade[flow],E01|E05|E06,R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01,P01|P02|P05|P07|P08|P09|P10,currency_raw|flow_raw|value_raw,customs_declarations.csv|erp_export.xlsx|manual_sheet.xlsx
1,V02,Total Imports,K02,Total Imports,Fact_Trade[value_tnd]|Fact_Trade[flow],E01|E05|E06,R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01,P01|P02|P05|P07|P08|P09|P10,currency_raw|flow_raw|value_raw,customs_declarations.csv|erp_export.xlsx|manual_sheet.xlsx
2,V03,Trade Balance,K03,Trade Balance,Fact_Trade[value_tnd]|Fact_Trade[flow],E01|E05|E06,R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01,P01|P02|P05|P07|P08|P09|P10,currency_raw|flow_raw|value_raw,customs_declarations.csv|erp_export.xlsx|manual_sheet.xlsx


## B.4 RAG — real retrieval with ChromaDB + sentence-transformers

Two collections are embedded with `all-MiniLM-L6-v2`:
1. **`trade_glossary`** — domain knowledge (HS codes, ISO codes, FX, KPI definitions, audit rules).
2. **`audit_trail`** — every event written by the agents (so the Auditor can answer free-form "Why?" questions).

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

GLOSSARY = [
    ("G-HS", "HS code", "The Harmonized System (HS) classifies traded goods. The first 6 digits are common to all countries (chapter, heading, subheading). A valid HS6 code has exactly 6 digits; punctuation such as 1509.10 or 1509-10 is only formatting. A code with fewer or more digits cannot be mapped to a product and must be reviewed."),
    ("G-ISO", "ISO country codes", "ISO 3166-1 alpha-2 codes (TN, FR, IT) give each country one unambiguous two-letter code. Customs data often contains local names (Tunisie, Italie), abbreviations (UK, USA) or typos, so country names must be normalised to ISO codes before any aggregation by partner country."),
    ("G-FX", "Currency conversion", "Trade values are declared in several currencies (EUR, USD, TND). To compare or add them, each value is converted to one reporting currency, here the Tunisian dinar (TND), using a documented exchange-rate table. The rate used must be recorded for every conversion so the figure can be reproduced in an audit."),
    ("G-DUP", "Duplicate declarations", "A customs declaration has a unique identifier. If the same declaration_id appears twice, for example because the same file was loaded twice, the value is counted twice and inflates exports or imports. Duplicates are removed keeping the first occurrence, and each removal is logged."),
    ("G-TRADE-BALANCE", "Trade balance", "Trade balance = exports minus imports over a period. A positive balance is a surplus, a negative one is a deficit. It depends on the same value column and the same cleaning rules as total exports and total imports."),
    ("G-COVERAGE", "Coverage ratio", "Coverage ratio = exports divided by imports. A value above 1 means exports pay for all imports; below 1 means imports exceed exports. Customs and finance teams use it to judge external balance."),
    ("G-YOY", "Year-over-year growth", "Year-over-year (YoY) growth = (current year minus previous year) divided by previous year. In DAX it is computed with SAMEPERIODLASTYEAR. It needs complete, consistently cleaned data for both years, otherwise growth reflects data errors instead of trade."),
    ("G-AUDIT", "Customs audit traceability", "In a customs audit the company must justify every reported figure. This requires lineage from the dashboard number back to the declarations: which rows were used, which corrections were applied, with which rule, by whom, and whether a human approved it."),
    ("G-HUMAN", "Human validation policy", "Corrections proposed by heuristics or language models must remain under human control. Low-confidence changes (fuzzy matches, negative values, missing countries) are queued for a human reviewer. Until approved, the status is pending and the figure should be considered provisional."),
    ("G-QUARANTINE", "Quarantine of rows", "Rows that cannot be repaired automatically, such as a missing partner country or an invalid HS code, are quarantined: they stay in quarantine.csv and are excluded from the analytical model, so reported totals are lower than raw totals and the exclusion is documented."),
    ("G-DAX", "DAX measures", "DAX is the Power BI formula language. CALCULATE changes the filter context of an aggregation, for example summing value_tnd only where flow is Export. DIVIDE performs a safe division that avoids divide-by-zero errors."),
    ("G-ZSCORE", "Anomaly detection with z-score", "A z-score measures how many standard deviations a value is from the mean. An absolute z-score of at least 2 is a common threshold to flag an unusual month. An anomaly may be a real shipment or a data error, so it must be checked before it is reported."),
    ("G-CONCENTRATION", "Concentration risk", "Concentration risk appears when a large share of exports depends on one product or one partner country, or imports on one supplier. Thresholds such as 30% for a product or 25% for a supplier trigger a recommendation to diversify."),
    ("G-STAR", "Star schema", "A star schema has one fact table (Fact_Trade, one row per declaration) linked by surrogate keys to dimension tables (Dim_Country, Dim_Product, Dim_Date). Power BI models perform best with this structure."),
    ("G-INCOTERMS", "Incoterms", "Incoterms (FOB, CIF, EXW) define who pays transport and insurance. Values declared under different Incoterms are not directly comparable unless adjusted, which is a known limitation when comparing declared trade values."),
    ("G-DATE", "Date formats", "Dates in trade files appear as 2024-03-15, 15/03/2024 or Mar 15, 2024. Day/month ambiguity is a classic error source, so all dates are parsed with an explicit format and stored as ISO 8601."),
]
glossary_df = pd.DataFrame(GLOSSARY, columns=["id", "title", "text"])
save(glossary_df, "trade_glossary.csv")

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="sentence-transformers/all-MiniLM-L6-v2")
chroma = chromadb.PersistentClient(path=f"{OUT}/chroma")

def build_collection(name, ids, docs, metas):
    try:
        chroma.delete_collection(name)
    except Exception:
        pass
    col = chroma.create_collection(name, embedding_function=embed_fn, metadata={"hnsw:space": "cosine"})
    for i in range(0, len(ids), 500):
        col.add(ids=ids[i:i + 500], documents=docs[i:i + 500], metadatas=metas[i:i + 500])
    return col

col_gloss = build_collection("trade_glossary", glossary_df.id.tolist(), (glossary_df.title + ". " + glossary_df.text).tolist(),
                             [{"title": t} for t in glossary_df.title])
col_trail = build_collection("audit_trail", trail.event_id.tolist(), trail.summary.tolist(),
                             [{"agent": a, "event_type": t, "subject": str(s)} for a, t, s in zip(trail.agent, trail.event_type, trail.subject)])
print("indexed:", {"trade_glossary": col_gloss.count(), "audit_trail": col_trail.count()})

def retrieve(query, col, k=3, where=None):
    res = col.query(query_texts=[query], n_results=k, where=where)
    return [dict(id=i, text=d, distance=round(s, 3), meta=m)
            for i, d, s, m in zip(res["ids"][0], res["documents"][0], res["distances"][0], res["metadatas"][0])]

for h in retrieve("why was the exchange rate used to convert euros", col_gloss, 2):
    print(f"[{h['id']}] dist={h['distance']}  {h['text'][:110]}...")

  saved bi_flow_outputs/trade_glossary.csv               16 rows x  3 cols


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexed: {'trade_glossary': 16, 'audit_trail': 2803}
[G-FX] dist=0.383  Currency conversion. Trade values are declared in several currencies (EUR, USD, TND). To compare or add them, ...
[G-COVERAGE] dist=0.803  Coverage ratio. Coverage ratio = exports divided by imports. A value above 1 means exports pay for all imports...


## B.5 Real LLM — local Hugging Face model or Claude API

The LLM is loaded once. If it cannot be loaded (no GPU memory, missing key), the Auditor falls back to an **evidence-only template** and says so in the output (`model` column).

In [ ]:
class LLM:
    def __init__(self, provider):
        self.provider = provider
        self.name = HF_MODEL if provider == "hf" else ANTHROPIC_MODEL

    def load(self):
        if self.provider == "hf":
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
            self.tok = AutoTokenizer.from_pretrained(HF_MODEL)
            self.model = AutoModelForCausalLM.from_pretrained(HF_MODEL, torch_dtype="auto", device_map="auto")
            self.model.eval()
        else:
            import anthropic
            try:
                from google.colab import userdata
                key = userdata.get("ANTHROPIC_API_KEY")
            except Exception:
                key = os.environ.get("ANTHROPIC_API_KEY")
            self.client = anthropic.Anthropic(api_key=key)
        return self

    def __call__(self, system, user, max_new_tokens=None):
        max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
        if self.provider == "hf":
            msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
            text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = self.tok(text, return_tensors="pt").to(self.model.device)
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            return self.tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        r = self.client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_new_tokens, temperature=0,
                                        system=system, messages=[{"role": "user", "content": user}])
        return r.content[0].text.strip()

try:
    llm = LLM(LLM_PROVIDER).load()
    print(f"LLM ready: {llm.name} ({llm.provider})")
except Exception as e:
    llm = None
    print(f"WARNING: LLM could not be loaded ({type(e).__name__}: {e}).\n"
          "The Auditor will use the evidence-only template fallback; fix the cause and re-run this cell for real LLM text.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM ready: Qwen/Qwen2.5-1.5B-Instruct (hf)


## B.6 Evidence builders, grounded prompting and answer verification

1. Build an **evidence pack** (deterministic records + RAG hits), each item with an id.
2. The LLM must answer **only from the pack** and cite ids in brackets.
3. `ground_answer()` checks **every sentence** against the evidence: sentences with numbers or content words that are not in the evidence, or that are cut off, are removed; the citation of each kept sentence is attached from the evidence that supports it (small models often forget to cite). If most of an answer is unsupported, the LLM is retried once with feedback.

In [ ]:
ID_RE = r"\b(?:[A-Z]{1,2}\d{2,4}|R-[A-Z]+-\d{2}|G-[A-Z-]+)\b"

SYSTEM = """You are the BI Auditor / XAI agent of BI Flow, a multi-agent Power BI automation system for import/export trade data.
You explain results to a customs compliance manager who must justify every number in an audit.
STRICT RULES:
1. Use ONLY facts written in the EVIDENCE block. Never add outside knowledge, guesses, reasons or consequences that are not written there.
2. Write 3 to 5 short sentences. End EVERY sentence with the id of the evidence item it comes from in square brackets, for example: The rate used was 3.35 TND per EUR [C0100].
3. Copy numbers exactly as written in the evidence.
4. If the evidence does not contain what is asked, say precisely what is missing.
5. No headings, no bullet lists, no formulas, no closing remarks."""

CITE_RE = r"\[((?:[A-Z]{1,2}\d{2,4}|R-[A-Z]+-\d{2}|G-[A-Z-]+))\]"      # only real evidence ids count (DAX refs like [flow] do not)
STOP = set("a an the of to in on for and or is are was were be been being by with as that this these those it its from at which who than then so such "
           "into over under per via not no can could should would will may might also has have had there their they them what when where how why because "
           "while about between during each other more most some any all one two both only "
           "used use based shows show showing indicates indicating applied within context equals resulted result calculated calculates across due given made make makes "
           "provided provides according following specific specifically key main".split())

def stems(text):
    text = re.sub(ID_RE, " ", re.sub(r"\[[^\]]*\]", " ", text))
    return {w[:5] for w in re.findall(r"[a-z][a-z_\-]+", text.lower()) if w not in STOP and len(w) > 2}

def num_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(ID_RE, " ", text)
    out = []
    for t in re.findall(r"\d[\d,]*\.?\d*", text):
        t = t.replace(",", "").rstrip(".")
        try:
            out.append((float(t), "." in t))
        except ValueError:
            pass
    return out

def split_sentences(text):
    out = []
    for line in re.split(r"\n+", re.sub(r"\*\*|`", "", text)):
        line = re.sub(r"^\s*(?:[-*\u2022]|\d+[.)])\s+", "", line).strip()
        if not line or "\\" in line or line.startswith("$") or line.endswith(":"):
            continue
        out += [x.strip() for x in re.split(r"(?<=[.!?])\s+(?=[A-Z\[(\"'])", line) if x.strip()]
    return out

def ground_answer(answer, evidence, min_cov=0.5):
    """Sentence-level grounding: keep only sentences supported by the evidence, attach the supporting evidence ids."""
    ev_tok = {e["id"]: stems(e["text"]) for e in evidence}
    ev_all = set().union(*ev_tok.values()) if ev_tok else set()
    ev_nums = [v for e in evidence for v, _ in num_tokens(e["text"])]
    kept, dropped = [], []
    for sent in split_sentences(answer):
        own = [c for c in re.findall(CITE_RE, sent) if c in ev_tok]
        core = re.sub(r"\s*(?:" + CITE_RE + r"\s*)+", " ", sent).strip()
        core = re.sub(r"\s+([.!?])", r"\1", core)
        if not core or core[-1] not in ".!?":
            dropped.append((sent, "incomplete sentence (cut off)")); continue
        toks = stems(core)
        if not toks:
            dropped.append((sent, "no factual content")); continue
        bad = [v for v, d in num_tokens(core) if (v >= 10 or d) and not any(abs(v - x) <= max(0.06, 0.005 * abs(x)) for x in ev_nums)]
        cov = len(toks & ev_all) / len(toks)
        if bad:
            dropped.append((sent, f"number {bad[0]:g} not in evidence")); continue
        if cov < min_cov:
            dropped.append((sent, f"only {cov:.0%} of its content words are in the evidence")); continue
        ranked = sorted(((len(toks & t), i) for i, t in ev_tok.items()), reverse=True)
        cites = list(dict.fromkeys(own + [i for sc, i in ranked if sc >= 2][:2]))[:3] or [ranked[0][1]]
        kept.append(core + " " + " ".join(f"[{c}]" for c in cites))
    status = "failed" if not kept else ("verified" if not dropped else "filtered")
    return " ".join(kept), dict(status=status, kept=kept, dropped=dropped)

def template_answer(evidence):
    out = ""
    for e in evidence:
        if e["id"].startswith("G-"):
            continue
        piece = f"{e['text']} [{e['id']}] "
        if out and len(out) + len(piece) > 900:
            break
        out += piece
    return out.strip()

def answer_with_evidence(question, evidence):
    pack = "\n".join(f"[{e['id']}] {e['text']}" for e in evidence)
    prompt = f"QUESTION: {question}\n\nEVIDENCE:\n{pack}\n\nAnswer in 3 to 5 short sentences, each ending with an evidence id in brackets:"
    if not llm:                                          # no LLM loaded: plain evidence-only answer (already cited)
        text = template_answer(evidence)
        return text, dict(ok=True, status="template_fallback", n_dropped=0, cited=sorted(set(re.findall(CITE_RE, text))), issues=[])
    feedback, best = "", None
    for attempt in range(2):
        raw_text = llm(SYSTEM, prompt + feedback)
        text, info = ground_answer(raw_text, evidence)
        if best is None or len(info["kept"]) > len(best[1]["kept"]):
            best = (text, info)
        if info["status"] == "verified" or len(info["kept"]) > len(info["dropped"]):
            break
        why = "; ".join(f"'{s[:60]}...' ({r})" for s, r in info["dropped"][:2])
        feedback = f"\n\nYour previous answer contained unsupported or cut-off sentences: {why}. Write a shorter answer using only the evidence."
    text, info = best
    if info["status"] == "failed":                      # never return an empty answer: fall back to the raw evidence
        text, info = ground_answer(template_answer(evidence), evidence)
        info["status"] = "template_fallback"
    v = dict(ok=info["status"] != "failed", status=info["status"], n_dropped=len(info["dropped"]),
             cited=sorted(set(re.findall(CITE_RE, text))), issues=[r for _, r in info["dropped"]][:4])
    return text, v

def E(eid, text=None):
    return dict(id=eid, text=text or TRAIL[eid])

def glossary_evidence(query, k=2):
    return [dict(id=h["id"], text=h["text"]) for h in retrieve(query, col_gloss, k)]

# ---------- evidence for each XAI question ----------
def kpi_evidence(kpi_id, dimension="none"):
    ln = lineage_for(kpi_id, dimension)
    n = kpi_id[1:]
    vals = kpi_val[(kpi_val.kpi_id == kpi_id) & (kpi_val.scope_type == "all")].sort_values("period")
    ev_ = [E(kpi_id)]
    ev_.append(E(f"KV{n}", f"Published values of {kpi_id}: " + "; ".join(
        f"{r.period} = {fmt_kpi(kpi_id, r.value)} from {r.rows_used} rows" for r in vals.itertuples())))
    r_ = recon[(recon.kpi_id == kpi_id) & (recon.scope_type == "all")]
    ev_.append(E(f"RC{n}", f"Independent recomputation of {kpi_id} from clean_trade_data.csv: {(r_.status == 'MATCH').sum()} of {len(r_)} "
                           f"published overall values match; largest absolute difference {r_.abs_diff.max():.6f}."))
    last = int(vals.period.max())
    ps = prov_stats(subset_for(kpi_id, last))
    ev_.append(E(f"PV{n}", f"Provenance of {kpi_id} for {last}: {ps['rows']} rows feed the figure; {ps['normalised_rows']} of them only needed label or format "
                           f"normalisation, and {ps['judgment_rows']} received a judgment-based correction (fuzzy match or negative value) representing "
                           f"{ps['judgment_value_pct']}% of its value; {ps['pending_rows']} rows still have a correction pending human validation. "
                           f"Most frequent rules: {ps['top_rules']}."))
    ev_.append(E(f"LN{n}", f"Lineage of {kpi_id} {ln['kpi_name']}: fact columns {ln['fact_columns']}; ETL steps {ln['etl_steps']}; "
                           f"cleaning rules {ln['cleaning_rules']}; profiler issues {ln['profiler_issues']}; raw columns {ln['raw_columns']}; "
                           f"raw sources {ln['raw_sources']}."))
    for e_id in split(ln["etl_steps"]):
        if e_id in ("E05", "E01"):
            ev_.append(E(e_id))
    return ev_

def explain_correction(event_id):
    e = clog[clog.event_id == event_id].iloc[0]
    q = f"Why was the {e.column} of row {e.row_id} changed from '{e.before}' to '{e.after}'?"
    evid = [E(event_id), E(e.profiler_issue_id), E(e.rule_id), E("O02")] + glossary_evidence(f"{e.rule_description} {e.column}", 2)
    return q, evid

def explain_kpi(kpi_id):
    return f"Why does the KPI {kpi_id} exist, how is it computed and where does its figure come from?", kpi_evidence(kpi_id) + glossary_evidence(
        kpi_cat.loc[kpi_cat.kpi_id == kpi_id, "name"].iloc[0] + " " + kpi_cat.loc[kpi_cat.kpi_id == kpi_id, "description"].iloc[0], 1)

def insight_prov(i):
    q = f"flow == '{i.flow}' and year in {[int(y) for y in split(i.years)]}" + (f" and {i.filter_expr}" if i.filter_expr else "")
    ps = prov_stats(BASE.query(q))
    return E(f"PI{i.insight_id[1:]}", f"Provenance of insight {i.insight_id}: it uses {ps['rows']} rows; {ps['normalised_rows']} only needed label or format normalisation and "
                                      f"{ps['judgment_rows']} received a judgment-based correction ({ps['judgment_value_pct']}% of their value); "
                                      f"{ps['pending_rows']} have a correction pending human validation.")

def explain_insight(insight_id):
    i = ins[ins.insight_id == insight_id].iloc[0]
    kid = i.kpi_id
    kv = kpi_val[(kpi_val.kpi_id == kid) & (kpi_val.scope_type == "all")].sort_values("period").iloc[-1]
    return (f"Where does the insight {insight_id} come from?",
            [E(insight_id), E(kid), insight_prov(i), E(f"KV{kid[1:]}", f"Published {kid} for {kv.period} = {fmt_kpi(kid, kv.value)} from {kv.rows_used} rows."),
             E(f"RC{kid[1:]}", f"Independent recomputation of {kid} matched the published value."),
             ] + glossary_evidence(f"{i.method} {i.text}", 1))

def explain_recommendation(insight_id):
    i = ins[ins.insight_id == insight_id].iloc[0]
    return (f"How is the recommendation of insight {insight_id} justified?",
            [E(insight_id), E(f"RL{insight_id[1:]}", f"Decision rule for {insight_id}: {i.rule_text} Observed {i.metric_name} = {i.metric_value}; "
                                                       f"threshold = {i.threshold}; rule triggered = {'yes' if i.rule_triggered else 'no'}. Recommendation issued: {i.recommendation}")]
            + glossary_evidence(i.rule_text, 1))

## B.7 Run the explanations (the four XAI questions)

* *Why was this value corrected?* → `explain_correction`
* *Why this KPI?* → `explain_kpi`
* *Where does the insight come from?* → `explain_insight`
* *How is the recommendation justified?* → `explain_recommendation`

In [ ]:
import time
xai = []

def run(qtype, subject, builder):
    q, evid = builder(subject)
    t0 = time.time()
    text, v = answer_with_evidence(q, evid)
    xai.append(dict(question_type=qtype, subject_id=subject, question=q, answer=text, verified=v["ok"], status=v["status"], n_dropped=v["n_dropped"],
                    cited_ids="|".join(v["cited"]), verification_issues="; ".join(v["issues"]),
                    evidence_ids="|".join(e["id"] for e in evid), model=(llm.name if llm else "template-fallback"),
                    seconds=round(time.time() - t0, 1), timestamp=dt.datetime.now().isoformat(timespec="seconds")))
    print(f"\n[{qtype} | {subject}] status={v['status']} (removed {v['n_dropped']} unsupported sentences)\nQ: {q}\nA: {text}")

# one correction per rule: human-validation cases first
sel = []
for rid, g in clog.groupby("rule_id"):
    sel.append(g.sort_values(["needs_human_validation", "confidence"], ascending=[False, True]).iloc[0])
sel = sorted(sel, key=lambda r: (not r.needs_human_validation, r.rule_id))[:N_CORR_EXPLAIN]

for r in sel:                                  run("correction", r.event_id, explain_correction)
for kid in kpi_cat.kpi_id[:N_KPI_EXPLAIN]:     run("kpi", kid, explain_kpi)
for iid in ins.insight_id[:N_INSIGHT_EXPLAIN]:
    run("insight", iid, explain_insight)
    run("recommendation", iid, explain_recommendation)

xai_df = pd.DataFrame(xai)
save(xai_df, "xai_explanations.csv")
print("\nStatus of the", len(xai_df), "answers:", xai_df.status.value_counts().to_dict())


[correction | C0148] status=filtered (removed 3 unsupported sentences)
Q: Why was the country of row 32 changed from 'Frnace' to 'FR'?
A: This misspelled country name was then matched to the closest valid alias, which is "FR". [C0148] [R-CTRY-02] Once reviewed, the correct ISO country code "TN" (for Tunisia) was applied to ensure accurate representation in the dataset. [G-ISO] [G-HUMAN]

[correction | C0022] status=verified (removed 0 unsupported sentences)
Q: Why was the country of row 5 changed from '(missing)' to 'UNK'?
A: The country of row 5 was changed from '(missing)' to 'UNK' because the rule R-CTRY-03 identified that there was no partner country in the declaration, leading to the recommendation to set the country to 'UNK' and quarantine the row. [C0022] [R-CTRY-03] This change addresses the issue detected by Data Profiler , ensuring that only relevant countries contribute to the analysis. [P02] [R-CTRY-03]

[correction | C0630] status=verified (removed 0 unsupported sentences

## B.8 Interactive "Why?" interface — free-form questions with RAG

`ask_auditor(question)` combines (1) exact lookups when the question names a row or an id, (2) semantic retrieval over the audit trail and the glossary, and (3) KPI lineage when a KPI is mentioned.

In [ ]:
def ask_auditor(question, k=4, show=True):
    evid, seen = [], set()
    def add(items):
        for it in items:
            if it and it["id"] not in seen:
                seen.add(it["id"]); evid.append(it)

    m = re.search(r"row\s*#?\s*(\d+)", question, re.I)
    if m:
        add([E("O02")])
        for r in clog[clog.row_id == m.group(1)].itertuples():
            if r.rule_id != "R-FX-01" or "convert" in question.lower() or "tnd" in question.lower():
                add([E(r.event_id)])
    for i in re.findall(ID_RE, question):
        if i in TRAIL: add([E(i)])
    ql = question.lower()
    for r in kpi_cat.itertuples():
        if r.kpi_id.lower() in ql or r.name.lower() in ql:
            add(kpi_evidence(r.kpi_id))
    if not m:                                   # a named row is answered from its own events only
        add([dict(id=h["id"], text=h["text"]) for h in retrieve(question, col_trail, k)])
    add(glossary_evidence(question, 2))
    text, v = answer_with_evidence(question, evid[:14])
    if show:
        print(f"Q: {question}\nA: {text}\n   status={v['status']}  removed={v['n_dropped']}  cited={v['cited']}\n")
    return text, v

fuzzy = clog[clog.rule_id == "R-CTRY-02"]
sample_row = (fuzzy.iloc[0] if len(fuzzy) else clog[clog.rule_id != "R-FX-01"].iloc[0]).row_id
ask_auditor(f"Why was the country of row {sample_row} changed and has a human approved it?")
ask_auditor("Which agents touched the Trade Balance figure and how can I trace it back to the raw files?")
ask_auditor("Why does the coverage ratio matter for a customs audit?")

# Uncomment to chat with the Auditor:
# while True:
#     q = input("Ask the Auditor (empty to stop): ").strip()
#     if not q: break
#     ask_auditor(q)

Q: Why was the country of row 32 changed and has a human approved it?
A: Row 32 was changed because the flow label was written in a non-standard way, which triggered a rule change. [C0147] [C0148] This change was approved by the system without further review. [G-HUMAN] [C0148] After human validation, this change was also approved. [O02] [G-HUMAN]
   status=filtered  removed=2  cited=['C0147', 'C0148', 'G-HUMAN', 'O02']

Q: Which agents touched the Trade Balance figure and how can I trace it back to the raw files?
A: ETL Steps: The data undergoes several ETL processes including filtering out quarantined rows, building Fact_Trade, and cleaning rules such as R-CTRY-01, R-CUR-01, etc., ensuring consistency across different datasets like custom trade declarations, ERP export sheets, and manual entries. [LN03] [E05] Provenance: The dataset includes multiple raw columns like currency_raw, flow_raw, and value_raw, sourced from various raw files including customs declarations, ERP export sheets

('The coverage ratio matters for a customs audit because it provides insight into how much of the imported goods can be covered by exported goods. [G-COVERAGE] [K04]',
 {'ok': True,
  'status': 'filtered',
  'n_dropped': 3,
  'cited': ['G-COVERAGE', 'K04'],
  'issues': ['only 36% of its content words are in the evidence',
   'only 42% of its content words are in the evidence',
   'only 33% of its content words are in the evidence']})

## B.9 Human review queue and the final XAI report

In [ ]:
queue = clog[clog.needs_human_validation].copy()
queue["priority"] = np.where(queue.rule_id.isin(QUARANTINE_RULES), "high", np.where(queue.confidence < 0.8, "medium", "low"))
queue = queue.sort_values(["approval_status", "priority"], ascending=[False, True])[
    ["event_id", "row_id", "declaration_id", "column", "before", "after", "rule_id", "confidence", "approval_status", "approver", "priority", "reason"]]
save(queue, "human_review_queue.csv")
print(queue.approval_status.value_counts().to_dict())

  saved bi_flow_outputs/human_review_queue.csv           34 rows x 12 cols
{'pending': 32, 'approved': 2}


In [ ]:
from IPython.display import Markdown, display

def md_table(df):
    return df.to_markdown(index=False) if hasattr(df, "to_markdown") else df.to_string(index=False)

try:
    import tabulate  # noqa: F401
except Exception:
    get_ipython().run_line_magic("pip", "install -q tabulate")

run_id = orch.run_id.iloc[0]
cl_nofx = clog[clog.rule_id != "R-FX-01"]
parts = [f"# XAI Report — BI Flow run {run_id}\n",
         f"*Generated {dt.datetime.now().isoformat(timespec='seconds')} by the BI Auditor / XAI agent. "
         f"Explanations written by `{llm.name if llm else 'template-fallback'}` and verified against the evidence.*\n",
         "## 1. Run summary\n", md_table(orch[["step_no", "agent", "status", "retries", "human_validation_required", "human_validation_status"]]),
         "\n## 2. Data quality and corrections\n", md_table(prof[["issue_id", "issue_type", "column", "affected_rows", "pct_rows", "severity"]]),
         "\nCorrections by rule (excluding currency conversion):\n",
         md_table(cl_nofx.groupby("rule_id").agg(corrections=("event_id", "count"), pending=("approval_status", lambda s: (s == "pending").sum())).reset_index()),
         f"\nRows quarantined (excluded from the model): **{(clean.quality_flag == 'quarantined').sum()}**; rows pending review: **{(clean.quality_flag == 'pending_review').sum()}**.\n",
         "## 3. Reconciliation of published figures\n",
         f"{(recon.status == 'MATCH').sum()} of {len(recon)} published figures were recomputed independently and match.\n",
         md_table(recon[recon.scope_type == "all"].assign(published=lambda d: d.published.round(4), recomputed=lambda d: d.recomputed.round(4))
                  [["kpi_id", "period", "published", "recomputed", "status"]])]

def section(title, qtype):
    out = [f"\n## {title}\n"]
    for r in xai_df[xai_df.question_type == qtype].itertuples():
        flag = {"verified": "verified", "filtered": f"verified after removing {r.n_dropped} unsupported sentence(s)",
                "template_fallback": "evidence-only fallback (LLM answer unusable)"}.get(r.status, r.status)
        out.append(f"**{r.subject_id}** — *{r.question}*\n\n{r.answer}\n\n`{flag}; evidence: {r.evidence_ids}`\n")
    return out

parts += section("4. Why were values corrected?", "correction")
parts += section("5. Why these KPIs?", "kpi")
parts += section("6. Where do the insights come from?", "insight")
parts += section("7. How are the recommendations justified?", "recommendation")
parts += ["\n## 8. Lineage (visual → KPI → source)\n",
          md_table(lineage[["visual_id", "visual_title", "kpi_id", "etl_steps", "cleaning_rules", "profiler_issues"]]),
          "\n## 9. Human review queue (top 10)\n", md_table(queue.head(10)[["event_id", "row_id", "column", "before", "after", "rule_id", "confidence", "approval_status", "priority"]]),
          "\n## 10. Limitations\n",
          "- Upstream agent outputs in this notebook are simulated; the Auditor logic is independent of that.\n"
          "- LLM text is verified for cited ids and numbers, not for every semantic nuance; the deterministic evidence is the source of truth.\n"
          "- Corrections still marked `pending` are provisional until a human approves them.\n"]
report = "\n".join(parts)
with open(f"{OUT}/xai_report.md", "w", encoding="utf-8") as fh:
    fh.write(report)
print(f"saved {OUT}/xai_report.md ({len(report):,} characters)")
display(Markdown(report))

saved bi_flow_outputs/xai_report.md (19,467 characters)


# XAI Report — BI Flow run RUN-2026-10-28-001

*Generated 2026-10-28T18:57:57 by the BI Auditor / XAI agent. Explanations written by `Qwen/Qwen2.5-1.5B-Instruct` and verified against the evidence.*

## 1. Run summary

|   step_no | agent                  | status   |   retries | human_validation_required   | human_validation_status      |
|----------:|:-----------------------|:---------|----------:|:----------------------------|:-----------------------------|
|         1 | Data Profiler          | success  |         0 | no                          |                              |
|         2 | Quality & Cleaning     | success  |         0 | yes                         | 2 of 34 approved, 32 pending |
|         3 | ETL / Data Engineering | success  |         1 | no                          |                              |
|         4 | Semantic & KPI         | success  |         0 | no                          |                              |
|         5 | BI Analyst             | success  |         0 | no                          |                              |
|         6 | Dashboard Generator    | success  |         0 | no                          |                              |

## 2. Data quality and corrections

| issue_id   | issue_type            | column         |   affected_rows |   pct_rows | severity   |
|:-----------|:----------------------|:---------------|----------------:|-----------:|:-----------|
| P01        | duplicate_declaration | declaration_id |              30 |       4.76 | high       |
| P02        | missing_country       | country_raw    |              18 |       2.86 | high       |
| P03        | country_spelling      | country_raw    |             490 |      77.78 | medium     |
| P04        | hs_format             | hs_code_raw    |             404 |      64.13 | low        |
| P05        | hs_invalid            | hs_code_raw    |               5 |       0.79 | high       |
| P06        | date_format           | date_raw       |             324 |      51.43 | low        |
| P07        | flow_label            | flow_raw       |             512 |      81.27 | low        |
| P08        | currency_label        | currency_raw   |             473 |      75.08 | low        |
| P09        | negative_value        | value_raw      |               5 |       0.79 | high       |
| P10        | mixed_currency        | currency_raw   |             596 |      94.6  | medium     |

Corrections by rule (excluding currency conversion):

| rule_id   |   corrections |   pending |
|:----------|--------------:|----------:|
| R-CTRY-01 |           451 |         0 |
| R-CTRY-02 |            16 |         6 |
| R-CTRY-03 |            16 |        16 |
| R-CUR-01  |           450 |         0 |
| R-DATE-01 |           308 |         0 |
| R-DUP-01  |            30 |         0 |
| R-FLOW-01 |           486 |         0 |
| R-HS-01   |           385 |         0 |
| R-HS-02   |             5 |         5 |
| R-VAL-01  |             5 |         5 |

Rows quarantined (excluded from the model): **21**; rows pending review: **11**.

## 3. Reconciliation of published figures

51 of 51 published figures were recomputed independently and match.

| kpi_id   |   period |        published |       recomputed | status   |
|:---------|---------:|-----------------:|-----------------:|:---------|
| K01      |     2023 |      1.89905e+07 |      1.89905e+07 | MATCH    |
| K02      |     2023 |      1.8e+07     |      1.8e+07     | MATCH    |
| K03      |     2023 | 990463           | 990463           | MATCH    |
| K04      |     2023 |      1.055       |      1.055       | MATCH    |
| K01      |     2024 |      2.22516e+07 |      2.22516e+07 | MATCH    |
| K02      |     2024 |      2.16799e+07 |      2.16799e+07 | MATCH    |
| K03      |     2024 | 571711           | 571711           | MATCH    |
| K04      |     2024 |      1.0264      |      1.0264      | MATCH    |
| K05      |     2024 |      0.1717      |      0.1717      | MATCH    |

## 4. Why were values corrected?

**C0148** — *Why was the country of row 32 changed from 'Frnace' to 'FR'?*

This misspelled country name was then matched to the closest valid alias, which is "FR". [C0148] [R-CTRY-02] Once reviewed, the correct ISO country code "TN" (for Tunisia) was applied to ensure accurate representation in the dataset. [G-ISO] [G-HUMAN]

`verified after removing 3 unsupported sentence(s); evidence: C0148|P03|R-CTRY-02|O02|G-HUMAN|G-ISO`

**C0022** — *Why was the country of row 5 changed from '(missing)' to 'UNK'?*

The country of row 5 was changed from '(missing)' to 'UNK' because the rule R-CTRY-03 identified that there was no partner country in the declaration, leading to the recommendation to set the country to 'UNK' and quarantine the row. [C0022] [R-CTRY-03] This change addresses the issue detected by Data Profiler , ensuring that only relevant countries contribute to the analysis. [P02] [R-CTRY-03]

`verified; evidence: C0022|P02|R-CTRY-03|O02|G-QUARANTINE|G-ISO`

**C0630** — *Why was the hs_code of row 143 changed from '08041' to 'INVALID'?*

Row 143's original HS code '08041', which should have been 6 digits long according to the Harmonized System rules, was incorrectly formatted as having only 5 digits. [G-HS] [R-HS-02] This discrepancy led to the change in the HS code to 'INVALID' as per Rule R-HS-02, which requires setting the code to INVALID if it doesn't meet the 6-digit requirement. [C0630] [R-HS-02] The human validation process noted this issue and recommended quarantining the row due to its invalid format. [R-HS-02] [C0630]

`verified; evidence: C0630|P05|R-HS-02|O02|G-HS|G-QUARANTINE`

**C0042** — *Why was the value of row 9 changed from '-30295.62' to '30295.62'?*

The value of row 9 (-30295.62) was changed to 30295.62 because the original negative value was deemed implausible according to the profiling tool. [P09] [C0042] [R-VAL-01] This change uses the absolute value pending further human review , indicating that the initial negative value needs verification before finalizing the trade balance calculation. [C0042] [G-TRADE-BALANCE] [G-HUMAN]

`verified after removing 1 unsupported sentence(s); evidence: C0042|P09|R-VAL-01|O02|G-HUMAN|G-TRADE-BALANCE`

**C0002** — *Why was the country of row 1 changed from 'Turquie' to 'TR'?*

This error was corrected by changing the country to its correct spelling, 'TR', using the alias reference table. [R-CTRY-01] [C0002]

`verified after removing 3 unsupported sentence(s); evidence: C0002|P03|R-CTRY-01|O02|G-ISO|G-HS`

**C0004** — *Why was the currency of row 1 changed from 'eur' to 'EUR'?*

The currency of row 1 was changed from 'eur' to 'EUR' because the original currency label was not recognized as an ISO-4217 code. [C0004] [R-CUR-01] This change was made through the rule R-CUR-01, which normalizes the currency label to its ISO-4217 code. [R-CUR-01] [C0004] The human validation requirement indicates that this change needed further review and approval. [O02] [C0004]

`verified; evidence: C0004|P08|R-CUR-01|O02|G-FX|G-ISO`


## 5. Why these KPIs?

**K01** — *Why does the KPI K01 exist, how is it computed and where does its figure come from?*

The KPI K01 Total Exports measures the total exports based on the SUM function applied to the Fact_Trade[value_tnd] column within the context of Fact_Trade[flow] being equal to "Export". [K01] [LN01] This computation relies on the sum of exported values in TND. [K01] [LN01] The figure originates from the published values of K01 for both 2023 and 2024, which were derived from the independent recomputation process using clean_trade_data.csv. [RC01] [LN01] The provenance indicates that most of these figures align closely with the published values, with minor discrepancies due to corrections made during the analysis phase. [PV01] [RC01] The lineage shows that the calculation involves various ETL steps, including filtering out quarantined rows, building Fact_Trade, and applying currency conversions, ensuring accuracy across different datasets and sources. [LN01] [E05]

`verified; evidence: K01|KV01|RC01|PV01|LN01|E01|E05|G-FX`

**K02** — *Why does the KPI K02 exist, how is it computed and where does its figure come from?*

This computation relies on the SUM function applied to the Fact_Trade[value_tnd] column within the context of Fact_Trade[flow] being equal to "Import". [LN02] [K02] The figure originates from the sum of import values expressed in TND across all valid entries in the Fact_Trade dataset. [K02] [PV02]

`verified after removing 1 unsupported sentence(s); evidence: K02|KV02|RC02|PV02|LN02|E01|E05|G-FX`

**K03** — *Why does the KPI K03 exist, how is it computed and where does its figure come from?*

The KPI K03 Trade Balance measures the net flow of goods between countries based on the value of exports and imports. [K03] [G-TRADE-BALANCE] It's calculated as Total Exports minus Total Imports using the Fact_Trade table, which includes both quantity and value information. [K03] [G-TRADE-BALANCE]

`verified after removing 2 unsupported sentence(s); evidence: K03|KV03|RC03|PV03|LN03|E01|E05|G-TRADE-BALANCE`

**K04** — *Why does the KPI K04 exist, how is it computed and where does its figure come from?*

The KPI K04 Coverage Ratio measures the extent to which exports cover imports within a given period. [K04] [G-COVERAGE] The figure originates from the Fact_Trade table, specifically the columns [Total Exports] and [Total Imports], providing insights into the economic balance between countries based on their export and import activities. [K04] [G-COVERAGE]

`verified after removing 1 unsupported sentence(s); evidence: K04|KV04|RC04|PV04|LN04|E01|E05|G-COVERAGE`

**K05** — *Why does the KPI K05 exist, how is it computed and where does its figure come from?*

This metric is calculated using the formula provided in the Evidence : VAR cur = [Total Exports] VAR prev = CALCULATE([Total Exports], SAMEPERIODLASTYEAR(Dim_Date[date])) RETURN DIVIDE(cur - prev, prev). [K05] [G-YOY] The result depends on the Fact_Trade table's value_tnd, flow, and Dim_Date[year] fields, reflecting the actual export figures and their corresponding dates. [LN05] [K05] The figure originates from 320 rows of exported data, which include some corrections based on fuzzy matches or negative values, totaling 3.4% of the original value. [PV05] [RC05] These corrections were made according to specific rules such as 'R-FLOW-01', 'R-CTRY-01', and 'R-CUR-01'. [PV05] The lineage of this KPI includes various ETL steps, cleansing rules, and profiling issues, ensuring the accuracy of the data before computation. [LN05] [G-YOY]

`verified after removing 1 unsupported sentence(s); evidence: K05|KV05|RC05|PV05|LN05|E01|E05|G-YOY`


## 6. Where do the insights come from?

**I01** — *Where does the insight I01 come from?*

This specific insight shows a decrease in exports to Spain by 17.4%, reflecting a significant drop in total exports to this market compared to the previous year. [I01] [G-YOY] The rule triggered due to the substantial negative percentage change suggests that further investigation into the commercial relationship with Spain is recommended, especially considering the threshold set at -10% for major partners' YoY changes. [I01] [G-YOY]

`verified after removing 1 unsupported sentence(s); evidence: I01|K01|PI01|KV01|RC01|G-YOY`

**I02** — *Where does the insight I02 come from?*

Insight I02 originates from the analysis of the top products exported in 2024 using the method top_share_by_product, which filters out all but the top 39 products based on their share of total export value. [I02] [K01] This insight identifies Virgin olive oil as the leading export product with 40.0% of the export value, triggering a rule to flag potential concentration risk due to its high dependency on this single product. [I02] [G-CONCENTRATION] The threshold of 30% has been met, indicating a concentration risk scenario where reliance on Virgin olive oil could be reduced through diversification efforts. [I02] [G-CONCENTRATION]

`verified; evidence: I02|K01|PI02|KV01|RC01|G-CONCENTRATION`

**I03** — *Where does the insight I03 come from?*

Insight I03 originates from the anomaly detection process using the z-score method. [I03] [G-ZSCORE] The metric calculated has an absolute z-score of 2.91, which exceeds the threshold of 2. [I03] [G-ZSCORE] This indicates that the export amount in July 2024 stands out compared to the monthly averages. [I03] [G-ZSCORE]

`verified after removing 1 unsupported sentence(s); evidence: I03|K01|PI03|KV01|RC01|G-ZSCORE`

**I04** — *Where does the insight I04 come from?*

Insight I04 originates from the analysis of the Top Supplier Share by Supplier metric applied to the Import flow in 2024, focusing specifically on suppliers from China. [I04] [G-CONCENTRATION] This insight identifies China as the top importer contributing 27.4% of total import value, triggering a recommendation to diversify import sources due to high concentration risk. [I04] [G-CONCENTRATION]

`verified; evidence: I04|K02|PI04|KV02|RC02|G-CONCENTRATION`


## 7. How are the recommendations justified?

**I01** — *How is the recommendation of insight I01 justified?*

The recommendation for insight I01 is justified because the observed export YoY percentage (-17.36) falls below the recommended threshold of -10%, indicating a significant decline in exports to Spain compared to the previous year. [RL01] [I01] This suggests potential issues with the commercial relationship with Spain and highlights the need for further investigation into diversifying away from this major partner. [RL01] [I01]

`verified after removing 1 unsupported sentence(s); evidence: I01|RL01|G-CONCENTRATION`

**I02** — *How is the recommendation of insight I02 justified?*

Insight I02 identifies Virgin olive oil as the top export product in 2024, accounting for 40.0% of the total export value. [I02] [RL02] The decision rule for I02 flags this situation and recommends reducing dependence on Virgin olive oil by exploring alternative export products. [RL02] [I02] This recommendation aligns with the identified concentration risk based on the observed top_product_share_pct being above the threshold of 30%. [RL02] [I02]

`verified after removing 1 unsupported sentence(s); evidence: I02|RL02|G-CONCENTRATION`

**I03** — *How is the recommendation of insight I03 justified?*

Insight I03 recommends checking large declarations of July 2024 due to the high z-score of 2.91 compared to the monthly average. [I03] [RL03] This indicates potential anomalies such as one-off shipments or data errors. [RL03] [I03] The decision rule for this insight flags months where the absolute z-score is at least 2, which has been met here. [RL03] [I03]

`verified after removing 1 unsupported sentence(s); evidence: I03|RL03|G-ZSCORE`

**I04** — *How is the recommendation of insight I04 justified?*

Insight I04 identifies China as the top import supplier of 2024, accounting for 27.4% of the total import value. [I04] [RL04] This concentration level triggers the decision rule, which recommends diversifying import sourcing away from China due to potential risks associated with high dependency on a single supplier. [RL04] [I04] The recommendation is based on the observed top supplier share percentage and the established threshold of 25%, leading to the conclusion that diversification is necessary to mitigate concentration risk. [G-CONCENTRATION] [RL04]

`verified; evidence: I04|RL04|G-CONCENTRATION`


## 8. Lineage (visual → KPI → source)

| visual_id   | visual_title                | kpi_id   | etl_steps       | cleaning_rules                                                                     | profiler_issues                 |
|:------------|:----------------------------|:---------|:----------------|:-----------------------------------------------------------------------------------|:--------------------------------|
| V01         | Total Exports               | K01      | E01|E05|E06     | R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01                     | P01|P02|P05|P07|P08|P09|P10     |
| V02         | Total Imports               | K02      | E01|E05|E06     | R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01                     | P01|P02|P05|P07|P08|P09|P10     |
| V03         | Trade Balance               | K03      | E01|E05|E06     | R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01                     | P01|P02|P05|P07|P08|P09|P10     |
| V04         | Coverage Ratio              | K04      | E01|E05|E06     | R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01                     | P01|P02|P05|P07|P08|P09|P10     |
| V05         | Export YoY Growth           | K05      | E01|E04|E05|E06 | R-CTRY-03|R-CUR-01|R-DATE-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01           | P01|P02|P05|P06|P07|P08|P09|P10 |
| V06         | Exports by partner country  | K01      | E01|E02|E05|E06 | R-CTRY-01|R-CTRY-02|R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01 | P01|P02|P03|P05|P07|P08|P09|P10 |
| V07         | Exports by product          | K01      | E01|E03|E05|E06 | R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-01|R-HS-02|R-VAL-01             | P01|P02|P04|P05|P07|P08|P09|P10 |
| V08         | Monthly exports             | K01      | E01|E04|E05|E06 | R-CTRY-03|R-CUR-01|R-DATE-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01           | P01|P02|P05|P06|P07|P08|P09|P10 |
| V09         | Imports by supplier country | K02      | E01|E02|E05|E06 | R-CTRY-01|R-CTRY-02|R-CTRY-03|R-CUR-01|R-DUP-01|R-FLOW-01|R-FX-01|R-HS-02|R-VAL-01 | P01|P02|P03|P05|P07|P08|P09|P10 |

## 9. Human review queue (top 10)

| event_id   |   row_id | column   | before    | after   | rule_id   |   confidence | approval_status   | priority   |
|:-----------|---------:|:---------|:----------|:--------|:----------|-------------:|:------------------|:-----------|
| C0022      |        5 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0060      |       13 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0074      |       16 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0098      |       21 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0288      |       63 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0630      |      143 | hs_code  | 08041     | INVALID | R-HS-02   |            0 | pending           | high       |
| C0696      |      159 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C0873      |      198 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C1177      |      268 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |
| C1533      |      351 | country  | (missing) | UNK     | R-CTRY-03 |            0 | pending           | high       |

## 10. Limitations

- Upstream agent outputs in this notebook are simulated; the Auditor logic is independent of that.
- LLM text is verified for cited ids and numbers, not for every semantic nuance; the deterministic evidence is the source of truth.
- Corrections still marked `pending` are provisional until a human approves them.


## B.10 Download everything

In [ ]:
shutil.make_archive("bi_flow_outputs", "zip", OUT)
print("Created bi_flow_outputs.zip with:", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download("bi_flow_outputs.zip")
except Exception:
    print("(Not running in Colab: the zip is in the current folder.)")

Created bi_flow_outputs.zip with: ['analyst_insights.csv', 'audit_trail.csv', 'chroma', 'clean_trade_data.csv', 'cleaning_log.csv', 'cleaning_rules.csv', 'column_lineage.csv', 'dashboard_visuals.csv', 'dim_country.csv', 'dim_date.csv', 'dim_product.csv', 'etl_log.csv', 'fact_trade.csv', 'fx_rates.csv', 'human_review_queue.csv', 'kpi_catalog.csv', 'kpi_values.csv', 'lineage.csv', 'orchestrator_runs.csv', 'profiler_report.csv', 'quarantine.csv', 'raw_trade_data.csv', 'reconciliation.csv', 'trade_glossary.csv', 'xai_explanations.csv', 'xai_report.md']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>